# Olist KPI Analysis and Business Insights

This notebook analyzes the validated delivered-order dataset to measure overall business performance, monthly trends, and category- and product-level results. Metrics are calculated at the appropriate grain to prevent duplicated order-level values from multi-item orders.

## 1. Load and Validate the Analysis-Ready Dataset

The analysis-ready Parquet dataset is loaded and its row count, order coverage, key uniqueness, and delivered-order scope are confirmed before KPI calculations begin.


In [535]:
# Import required libraries and define the processed-data path
import numpy as np
import pandas as pd 
from pathlib import Path

Processed_Data_Path = Path("../data/processed")

print(Processed_Data_Path.resolve())


C:\Users\Hp\DataAnalyst\data\processed


In [536]:
# Load the validated delivered-order-item dataset for KPI analysis.
delivered_order_items = pd.read_parquet(Processed_Data_Path/"delivered_order_items_analysis_ready.parquet")

In [537]:
delivered_order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,flag_future_shipping_limit_date,customer_id,order_status,...,delivery_difference_days,delivery_calendar_difference_days,delivery_status,freight_burden_percentage,order_purchase_month,order_purchase_year,order_purchase_quarter,order_purchase_month_number,order_purchase_weekday,order_purchase_weekday_number
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,False,3ce436f183e68e07877b285a838db11a,delivered,...,-8.011250,-9,early,22.563667,2017-09,2017,3,9,Wednesday,2
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,False,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,-2.330278,-3,early,8.307628,2017-04,2017,2,4,Wednesday,2
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,False,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,-13.444954,-14,early,8.979899,2018-01,2018,1,1,Sunday,6
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,False,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,-5.435660,-6,early,98.460354,2018-08,2018,3,8,Wednesday,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,False,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,-15.303808,-16,early,9.074537,2017-02,2017,1,2,Saturday,5


In [538]:
# Validate the loaded dataset structure and delivered order-item grain.
print(f"Shape of delivered_order_items : {delivered_order_items.shape}")
print(f"Distinct order_id count : {delivered_order_items['order_id'].nunique()}")
print(f"Unique order_status value: {delivered_order_items['order_status'].unique().tolist()}")
print(f"Duplicated order_id-order_item_id combinations: {delivered_order_items[['order_id', 'order_item_id']].duplicated().sum()}")
print(f"Duplicate column names: {delivered_order_items.columns.duplicated().sum()}")

Shape of delivered_order_items : (110197, 74)
Distinct order_id count : 96478
Unique order_status value: ['delivered']
Duplicated order_id-order_item_id combinations: 0
Duplicate column names: 0


## 2. Create the Order-Level Analytical Dataset

The source dataset contains one row per delivered order item, so orders containing multiple items appear more than once. A separate order-level dataset is required for metrics such as delivered-order count, average order value, delivery performance, payment method, and review score. Item prices and freight values are aggregated by `order_id`, while attributes that are constant within each order are retained once.

### 2.1 Aggregate Item Measures by Order


In [539]:
# Aggregate delivered item counts, merchandise value, and freight value to the order
order_item_summary = delivered_order_items.groupby('order_id').agg(order_item_count = ('order_item_id', 'count'),
                                                                   order_merchandise_value = ('price', 'sum'),
                                                                   order_freight_value = ('freight_value', 'sum')).reset_index()
order_item_summary.head()

,order_id,order_item_count,order_merchandise_value,order_freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


## 2. Create the Order-Level Analysis Dataset

The source dataset has one row per delivered order item, so order-level information is repeated for orders containing multiple items. A separate dataset with one row per `order_id` is created to calculate order counts, averages, delivery rates, payment metrics, and review metrics without double-counting multi-item orders.

### 2.1 Aggregate Item Measures by Order

Item records are aggregated to calculate each order’s item count, merchandise value, and total freight value.


In [540]:
# Validate the order-level item aggregation against the source dataset.
print(f"order_item_summary shape: {order_item_summary.shape}")
print(f"Duplicated order_id count: {order_item_summary['order_id'].duplicated().sum()}")
print(f"Missing value count:\n{order_item_summary.isna().sum()}")
print(f"Sum of order_item_count: {order_item_summary['order_item_count'].sum()}")

if order_item_summary['order_merchandise_value'].sum() == delivered_order_items['price'].sum():
    print(f"sum of both order_merchandised_value and price in delivered_order_items are same: { order_item_summary['order_merchandise_value'].sum()}")
else:
    print(f"sum of order_merchandise_value and price in delivered_order_items are different -\nsum of order_merchandise_value { order_item_summary['order_merchandise_value'].sum()} \n sum of price in delivered_order_items {delivered_order_items['price'].sum()}")

if order_item_summary['order_freight_value'].sum() == delivered_order_items['freight_value'].sum():
    print(f"sum of both order_freight_value and freight_value in delivered_order_items are same: { order_item_summary['order_freight_value'].sum()}")
else:
    print(f"sum of order_freight_value and freight_value in delivered_order_items are different -\nsum of order_freight_value { order_item_summary['order_freight_value'].sum()} \n sum of freight_value in delivered_order_items {delivered_order_items['freight_value'].sum()}")

order_item_summary shape: (96478, 4)
Duplicated order_id count: 0
Missing value count:
order_id                   0
order_item_count           0
order_merchandise_value    0
order_freight_value        0
dtype: int64
Sum of order_item_count: 110197
sum of both order_merchandised_value and price in delivered_order_items are same: 13221498.11
sum of both order_freight_value and freight_value in delivered_order_items are same: 2198275.64


### 2.2 Add Order-Level Attributes

Customer, delivery, payment, review, and purchase-time attributes are constant within each order but are repeated across item rows. One copy of these attributes is retained for every `order_id` and combined with the aggregated item measures to create the final order-level analytical dataset.


In [541]:
# Retain one copy of the repeated order-level attributes for each delivered order.
order_identity_customer= ['order_id', 'customer_id', 'customer_unique_id', 'customer_city', 'customer_state', 'order_status']
orders_timestamp_columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
payment_summary_columns =['total_payment_value', 'payment_record_count', 'distinct_payment_type_count', 'maximum_payment_installments', 'flag_payment_quality_exception', 'dominant_payment_type', 'flag_missing_payment_record']
reviews_summary_columns = ['review_record_count', 'average_review_score', 'review_title_count', 'review_message_count', 'average_review_response_duration', 'flag_review_response_over_30_days', 'flag_missing_review_record']
delivery_features = ['delivery_difference_days', 'delivery_duration_days', 'delivery_status', 'delivery_calendar_difference_days']
time_features = ['order_purchase_month', 'order_purchase_year', 'order_purchase_quarter', 'order_purchase_month_number', 'order_purchase_weekday', 'order_purchase_weekday_number']

order_attributes_columns = order_identity_customer + orders_timestamp_columns + payment_summary_columns + reviews_summary_columns + delivery_features + time_features
order_attributes = delivered_order_items[order_attributes_columns]
order_attributes = order_attributes.drop_duplicates(subset='order_id').reset_index(drop=True)
print(f"order_attributes shape: {order_attributes.shape}")
order_attributes.head()

order_attributes shape: (96478, 35)


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,...,delivery_difference_days,delivery_duration_days,delivery_status,delivery_calendar_difference_days,order_purchase_month,order_purchase_year,order_purchase_quarter,order_purchase_month_number,order_purchase_weekday,order_purchase_weekday_number
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,...,-8.011250,7.614421,early,-9,2017-09,2017,3,9,Wednesday,2
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,...,-2.330278,16.216181,early,-3,2017-04,2017,2,4,Wednesday,2
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,...,-13.444954,7.948437,early,-14,2018-01,2018,1,1,Sunday,6
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,atibaia,SP,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,...,-5.435660,6.147269,early,-6,2018-08,2018,3,8,Wednesday,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,varzea paulista,SP,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,...,-15.303808,25.114352,early,-16,2017-02,2017,1,2,Saturday,5


In [542]:
# Validate the deduplicated order attributes before combining order-level tables.
print(f"Duplicated order_id count: {order_attributes['order_id'].duplicated().sum()}")
print(f"Missing order_id count: {order_attributes['order_id'].isna().sum()}")
print(f"Unique order_status values: {order_attributes['order_status'].unique().tolist()}")

order_ids_match = ( set(order_attributes['order_id']) == set(order_item_summary['order_id']))
print(f"Order ID sets match: {order_ids_match}")

Duplicated order_id count: 0
Missing order_id count: 0
Unique order_status values: ['delivered']
Order ID sets match: True


### 2.3 Combine Order Attributes and Item Measures

The deduplicated order attributes are joined with the aggregated item measures using `order_id`. Both datasets contain exactly one row per delivered order, so the join is validated as one-to-one.


In [543]:
# Create the final order-level analytical dataset.
delivered_orders = pd.merge(order_attributes, order_item_summary, on='order_id', how='left', validate='one_to_one')
print(f"delivered_orders shape: {delivered_orders.shape}")
delivered_orders.head()

delivered_orders shape: (96478, 38)


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,...,delivery_calendar_difference_days,order_purchase_month,order_purchase_year,order_purchase_quarter,order_purchase_month_number,order_purchase_weekday,order_purchase_weekday_number,order_item_count,order_merchandise_value,order_freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,...,-9,2017-09,2017,3,9,Wednesday,2,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,...,-3,2017-04,2017,2,4,Wednesday,2,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,...,-14,2018-01,2018,1,1,Sunday,6,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,atibaia,SP,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,...,-6,2018-08,2018,3,8,Wednesday,2,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,varzea paulista,SP,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,...,-16,2017-02,2017,1,2,Saturday,5,1,199.90,18.14


In [544]:
# Validate the final order-level analytical dataset and reconciled totals.
print(f"Duplicate order_id count: {delivered_orders['order_id'].duplicated().sum()}")
print(f"Missing values in:\n{delivered_orders[['order_item_count', 'order_merchandise_value', 'order_freight_value']].isna().sum()}")
print(f"Sum of order_item_count: {delivered_orders['order_item_count'].sum()}")
print(f"Total of merchandise value: {delivered_orders['order_merchandise_value'].sum()}")
print(f"Total freight value: {delivered_orders['order_freight_value'].sum()}")
print(f"Unique order_status values: {delivered_orders['order_status'].unique().tolist()}")

Duplicate order_id count: 0
Missing values in:
order_item_count           0
order_merchandise_value    0
order_freight_value        0
dtype: int64
Sum of order_item_count: 110197
Total of merchandise value: 13221498.11
Total freight value: 2198275.64
Unique order_status values: ['delivered']


#### Order-Level Dataset Validation Summary

The order-level analytical dataset contains 96,478 unique delivered orders and 38 columns. Its aggregated item count, merchandise value, and freight value reconcile exactly with the item-level dataset. This table will be used for order-based KPIs to prevent multi-item orders from receiving additional weight.

## 3. Calculate Overall Business KPIs

Overall KPIs summarize delivered-order volume, merchandise value, freight cost, delivery performance, payment coverage, and customer-review outcomes. Each metric is calculated from the dataset grain appropriate to its business meaning.

### 3.1 Calculate Commercial KPIs


In [545]:
# Calculate overall delivered-order, merchandise, item, and freight KPIs.
delivered_orders_count = delivered_orders['order_id'].nunique()
print(f"delivered_orders_count: {delivered_orders_count}")

delivered_items_count = (delivered_orders['order_item_count'].sum())
print(f"delivered_items_count: {delivered_items_count}")

GMV = delivered_orders['order_merchandise_value'].sum()
print(f"GMV: {GMV}")

average_merchandise_value_per_order = GMV/delivered_orders_count
print(f"average_merchandise_value_per_order: {round(average_merchandise_value_per_order,4)}")

average_items_per_order = delivered_items_count/delivered_orders_count
print(f"average_items_per_order: {round(average_items_per_order, 4)}")

total_freight_value = delivered_orders['order_freight_value'].sum()
print(f"total_freight_value: {total_freight_value}")

overall_freight_burden_percentage = (total_freight_value/GMV)*100
print(f"overall_freight_burden_percentage: {round(overall_freight_burden_percentage,4)}")

delivered_orders_count: 96478
delivered_items_count: 110197
GMV: 13221498.11
average_merchandise_value_per_order: 137.0416
average_items_per_order: 1.1422
total_freight_value: 2198275.64
overall_freight_burden_percentage: 16.6265


#### Commercial KPI Summary

The delivered-order dataset contains 96,478 orders and 110,197 items. Delivered merchandise generated a GMV of R$13.22 million, with an average merchandise value of R$137.04 per order and 1.14 items per order. Total freight charges reached R$2.20 million, equivalent to 16.63% of merchandise value.


### 3.2 Calculate Delivery-Performance KPIs

Delivery KPIs are calculated at the order level so that multi-item orders are counted once. Orders without an actual customer delivery timestamp are excluded from duration averages and percentage denominators and are reported separately.


In [546]:
# Calculate order-level delivery duration and delivery-status KPIs.

average_delivery_duration_days = delivered_orders['delivery_duration_days'].mean()
print(f"average_delivery_duration_days: {average_delivery_duration_days}")

early_orders_count = len(delivered_orders[delivered_orders['delivery_status']== 'early'])
print(f"early_orders_count: {early_orders_count}")

on_time_orders_count = len(delivered_orders[delivered_orders['delivery_status']== 'on_time'])
print(f"on_time_orders_count: {on_time_orders_count}")

late_orders_count = len(delivered_orders[delivered_orders['delivery_status']== 'late'])
print(f"late_orders_count: {late_orders_count}")

missing_delivery_status_count = len(delivered_orders[delivered_orders['delivery_status']== 'missing'])
print(f"missing_delivery_status_count: {missing_delivery_status_count}")

total_delivered_orders = len(delivered_orders)
valid_delivery_orders_count = total_delivered_orders - missing_delivery_status_count
print(f"valid_delivery_orders_count: {valid_delivery_orders_count}")

early_delivery_rate = (early_orders_count/valid_delivery_orders_count)*100
print(f"early_delivery_rate: {early_delivery_rate}")

on_time_delivery_rate = (on_time_orders_count/valid_delivery_orders_count)*100
print(f"on_time_delivery_rate: {on_time_delivery_rate}")

late_delivery_rate = (late_orders_count/valid_delivery_orders_count)*100
print(f"late_delivery_rate: {late_delivery_rate}")

overall_on_or_before_estimate_rate = ((early_orders_count + on_time_orders_count)/valid_delivery_orders_count)*100
print(f"overall_on_or_before_estimate_rate: {overall_on_or_before_estimate_rate}")

average_delivery_duration_days: 12.558217098051976
early_orders_count: 88644
on_time_orders_count: 1292
late_orders_count: 6534
missing_delivery_status_count: 8
valid_delivery_orders_count: 96470
early_delivery_rate: 91.88763346117965
on_time_delivery_rate: 1.3392764590027988
late_delivery_rate: 6.7730900798175595
overall_on_or_before_estimate_rate: 93.22690992018245


In [547]:
# Confirm that early, on-time, and late delivery rates cover all valid orders.
sum_of_rates = early_delivery_rate + on_time_delivery_rate + late_delivery_rate
rates_cover_all_valid_orders = np.isclose(sum_of_rates, 100)

print(f"Sum of delivery rates: {sum_of_rates}")
print(f"Rates cover all valid orders: {rates_cover_all_valid_orders}")

Sum of delivery rates: 100.00000000000001
Rates cover all valid orders: True


In [548]:
# Count delivered orders with an available Review record.
orders_with_reviews = len(delivered_orders[delivered_orders['flag_missing_review_record'] == False])
orders_with_reviews

95832

In [549]:
# Calculate the percentage of delivered orders with an available Review record.
review_coverage_rate = (orders_with_reviews/len(delivered_orders))*100
review_coverage_rate

99.33041729720765

### 3.3 Calculate Customer Review KPIs

Review KPIs are calculated at the order level so that each delivered order contributes once. Average review scores use only orders with an available Review record, while review coverage separately reports the proportion of delivered orders with review information.


In [550]:
# Calculate the average review score across delivered orders with Review records.
average_review_score  = delivered_orders['average_review_score'].mean()
print(f"average review score: {average_review_score }")

average review score: 4.156186520855942


In [551]:
# Calculate the count and rate of reviewed delivered orders with low review scores.
low_review_score_count = len(delivered_orders[delivered_orders['average_review_score'] <= 2])
print(f"low_review_score_count: {low_review_score_count}")
low_review_score_percentage = (low_review_score_count / orders_with_reviews) *100
print(f"low_review_score_percentage: {low_review_score_percentage}")

low_review_score_count: 12237
low_review_score_percentage: 12.769221136989731


### 3.3 Calculate Customer Review KPIs

Review KPIs are calculated at the order level. The average review score and low-review rate include only orders with an available Review record, while review coverage measures the proportion of all delivered orders that received a review.


In [552]:
# Calculate the count and rate of reviewed orders with response durations over 30 days.
reviewed_orders_with_response_durations_over_30_days = delivered_orders[(delivered_orders['flag_missing_review_record']== False) & (delivered_orders['flag_review_response_over_30_days']== True)]
print(f"count reviewed_orders_with_response_durations_over_30_days: {len(reviewed_orders_with_response_durations_over_30_days)}")

reviewed_orders_with_response_durations_over_30_days_percentage = round((len(reviewed_orders_with_response_durations_over_30_days)/(delivered_orders['flag_review_response_over_30_days'].notna().sum()))*100, 4)
print(f"reviewed_orders_with_response_durations_over_30_days_percentage: {reviewed_orders_with_response_durations_over_30_days_percentage}")

count reviewed_orders_with_response_durations_over_30_days: 645
reviewed_orders_with_response_durations_over_30_days_percentage: 0.6731


### 3.4 Calculate Payment KPIs

Payment KPIs are calculated at the order level to avoid repeating payment values across item rows. Orders without matching Payment records are reported separately and excluded from calculations that require payment information.


In [553]:
# Calculate the count and rate of delivered orders without Payment records.
delivered_orders_without_payment_records = delivered_orders[delivered_orders['flag_missing_payment_record']==True]
print(f"delivered_order_without_payment_records count: {len(delivered_orders_without_payment_records)}")

delivered_orders_without_payment_records_percentage = round(((len(delivered_orders_without_payment_records))/(delivered_orders['flag_missing_payment_record'].notna().sum()))*100, 5)
print(f"delivered_order_without_payment_records_percentage: {delivered_orders_without_payment_records_percentage}")

delivered_order_without_payment_records count: 1
delivered_order_without_payment_records_percentage: 0.00104


In [554]:
# Count delivered orders by dominant payment type.
delivered_orders.groupby('dominant_payment_type')['order_id'].count()

dominant_payment_type
boleto         19191
credit_card    72785
debit_card      1484
missing            1
voucher         3017
Name: order_id, dtype: int64

In [555]:
# Calculate the percentage distribution of dominant payment types among orders with Payment records.
delivered_orders_dominant_payment_types_without_missing = delivered_orders[delivered_orders['dominant_payment_type']!='missing']
dominant_payment_types_distribution_percentage = ((delivered_orders_dominant_payment_types_without_missing['dominant_payment_type'].value_counts())/(len(delivered_orders_dominant_payment_types_without_missing)))*100
print(f"dominant_payment_types_distribution_percentage: {dominant_payment_types_distribution_percentage}")
print(f"sum of dominant_payment_types_distribution_percentage: {dominant_payment_types_distribution_percentage.sum()}")

dominant_payment_types_distribution_percentage: dominant_payment_type
credit_card    75.442852
boleto         19.891788
voucher         3.127170
debit_card      1.538190
Name: count, dtype: float64
sum of dominant_payment_types_distribution_percentage: 100.00000000000001


In [556]:
# Calculate the count and rate of orders with payment-quality exceptions.
flag_payment_quality_exception_order= delivered_orders[delivered_orders['flag_payment_quality_exception'] == True]
print(f"flag_payment_quality_exception_order count: {len(flag_payment_quality_exception_order)}")

flag_payment_quality_exception_order_percentage = (len(flag_payment_quality_exception_order)/(delivered_orders['flag_payment_quality_exception'].notna().sum()))*100
print(f"flag_payment_quality_exception_order_percentage: {round(flag_payment_quality_exception_order_percentage,5)}")

flag_payment_quality_exception_order count: 82
flag_payment_quality_exception_order_percentage: 0.08499


In [557]:
# Calculate the average maximum number of payment installments per delivered order.
average_maximum_payment_installments = delivered_orders['maximum_payment_installments'].mean()
average_maximum_payment_installments

np.float64(2.928024295946184)

### 3.5 Create the Overall KPI Summary

The validated commercial, delivery, review, and payment metrics are consolidated into a single summary table for reporting, visualization, and final business interpretation.


In [558]:
# Create the overall KPI summary with commercial, delivery, review, and payment metrics.
overall_kpi_summary = pd.DataFrame({
    "metric":["Delivered Orders",
              "Delivered Items", 
              "GMV",
              "Average Merchandise Value per Order",
              "Average Items per Order",
              "Total Freight Value",
              "Overall Freight Burden Percentage",
              "Average Delivery Duration (Days)",
              "Early Delivery Rate (%)",
              "On-Time Delivery Rate (%)",
              "Late Delivery Rate (%)",
              "On or Before Estimate Rate (%)",
              "Missing Delivery Status Orders",
              "Reviewed Orders",
              "Review Coverage Rate (%)",
              "Average Review Score",
              "Low Review Score Orders",
              "Low Review Score Rate (%)",
              "Review Response Over 30 Days Orders",
              "Review Response Over 30 Days Rate (%)",
              "Orders Without Payment Records",
              "Missing Payment Record Rate (%)",
              "Average Maximum Payment Installments",
              "Payment Quality Exception Orders",
              "Payment Quality Exception Rate (%)"],

    "value":[delivered_orders_count,
             delivered_items_count, 
             GMV,
             average_merchandise_value_per_order,
             average_items_per_order,
             total_freight_value,
             overall_freight_burden_percentage,
             average_delivery_duration_days,
             early_delivery_rate,
             on_time_delivery_rate,
             late_delivery_rate,
             overall_on_or_before_estimate_rate,
             missing_delivery_status_count,
             orders_with_reviews,
             review_coverage_rate,
             average_review_score,
             low_review_score_count,
             low_review_score_percentage,
             len(reviewed_orders_with_response_durations_over_30_days),
             reviewed_orders_with_response_durations_over_30_days_percentage,
             len(delivered_orders_without_payment_records),
             delivered_orders_without_payment_records_percentage,
             average_maximum_payment_installments,
             len(flag_payment_quality_exception_order),
             flag_payment_quality_exception_order_percentage]

    })

overall_kpi_summary.style.format({"value": "{:,.2f}"})

,metric,value
0,Delivered Orders,"96,478.00"
1,Delivered Items,"110,197.00"
2,GMV,"13,221,498.11"
3,Average Merchandise Value per Order,137.04
4,Average Items per Order,1.14
5,Total Freight Value,"2,198,275.64"
6,Overall Freight Burden Percentage,16.63
7,Average Delivery Duration (Days),12.56
8,Early Delivery Rate (%),91.89
9,On-Time Delivery Rate (%),1.34


In [559]:
# Validate the overall KPI summary structure and values.
print(f"overall_kpi_summary shape: {overall_kpi_summary.shape}")
print(f"Duplicated metric names: {overall_kpi_summary['metric'].duplicated().sum()}")
print(f"Missing values in both columns: {(overall_kpi_summary.isna().sum()).sum()}")
print(f"Whether every value is numeric: {overall_kpi_summary['value'].dtype.kind in 'iufc'}")

overall_kpi_summary shape: (25, 2)
Duplicated metric names: 0
Missing values in both columns: 0
Whether every value is numeric: True


#### Overall KPI Findings

The delivered-order scope contains 96,478 orders and 110,197 items, generating R$13.22 million in merchandise value. Average merchandise value was R$137.04 per order, while freight represented 16.63% of GMV.

Orders took an average of 12.56 days to reach customers. Approximately 93.23% arrived on or before the estimated date, while 6.77% arrived late.

Review coverage was 99.33%, with an average score of 4.16 out of 5. However, 12.77% of reviewed orders received a low score of 2 or below. Payment data was available for all but one delivered order, and only 0.085% of evaluable orders contained a payment-quality exception.


## 4. Analyze Monthly Business Performance

Monthly trends are calculated at the order level to measure changes in delivered-order volume, item volume, GMV, average order value, freight cost, delivery performance, and customer satisfaction over time.

### 4.1 Create the Monthly Commercial Summary


In [560]:
# Aggregate monthly delivered-order, item, merchandise, and freight measures.

monthly_commercial_summary = delivered_orders.groupby('order_purchase_month').agg(monthly_order_count = ('order_id', 'count'),
                                                                                  monthly_item_count = ('order_item_count', 'sum'),
                                                                                  monthly_gmv = ('order_merchandise_value', 'sum'),
                                                                                  monthly_freight_value = ('order_freight_value', 'sum')).reset_index()

monthly_commercial_summary

,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value
0,2016-09,1,3,134.97,8.49
1,2016-10,265,313,40325.11,6165.55
2,2016-12,1,1,10.90,8.72
3,2017-01,750,913,111798.36,15684.01
4,2017-02,1653,1858,234223.40,37015.92
5,2017-03,2546,2897,359198.85,55132.10
6,2017-04,2303,2569,340669.68,50142.72
7,2017-05,3546,4004,489338.25,77513.15
8,2017-06,3135,3489,421923.37,68127.00
9,2017-07,3872,4416,481604.52,84694.56


In [561]:
# Validate the monthly commercial summary against the overall totals.
print(f"monthly_commercial_summary shape: {monthly_commercial_summary.shape}")
print(f"Duplicated order_purchase_month values: {monthly_commercial_summary['order_purchase_month'].duplicated().sum()}")
print(f"Missing values across the table: {(monthly_commercial_summary.isna().sum()).sum()}")
print(f"Sum of monthly_order_count: {monthly_commercial_summary['monthly_order_count'].sum()}")
print(f"Sum of monthly_item_count: {monthly_commercial_summary['monthly_item_count'].sum()}")
print(f"Sum of monthly_gmv: {monthly_commercial_summary['monthly_gmv'].sum()}")
print(f"Sum of monthly_freight_value: {monthly_commercial_summary['monthly_freight_value'].sum()}")

monthly_commercial_summary shape: (23, 5)
Duplicated order_purchase_month values: 0
Missing values across the table: 0
Sum of monthly_order_count: 96478
Sum of monthly_item_count: 110197
Sum of monthly_gmv: 13221498.11
Sum of monthly_freight_value: 2198275.64


In [562]:
# Identify calendar months with no delivered orders in the analysis period.
complete_month_range = pd.period_range(min(monthly_commercial_summary['order_purchase_month']), max(monthly_commercial_summary['order_purchase_month']), freq='M')
observed_month_range = monthly_commercial_summary['order_purchase_month'].unique()

month_range_difference = complete_month_range.difference(observed_month_range)
month_range_difference

PeriodIndex(['2016-11'], dtype='period[M]')

In [563]:
# Add months with no delivered orders to preserve a continuous monthly timeline.

monthly_commercial_summary = (monthly_commercial_summary.set_index('order_purchase_month').reindex(complete_month_range, fill_value=0).rename_axis('order_purchase_month').reset_index())
monthly_commercial_summary

,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value
0,2016-09,1,3,134.97,8.49
1,2016-10,265,313,40325.11,6165.55
2,2016-11,0,0,0.00,0.00
3,2016-12,1,1,10.90,8.72
4,2017-01,750,913,111798.36,15684.01
5,2017-02,1653,1858,234223.40,37015.92
6,2017-03,2546,2897,359198.85,55132.10
7,2017-04,2303,2569,340669.68,50142.72
8,2017-05,3546,4004,489338.25,77513.15
9,2017-06,3135,3489,421923.37,68127.00


In [564]:
# Validate the continuous monthly timeline and preserved commercial totals.
print(f"monthly_commercial_summary shape: {monthly_commercial_summary.shape}")
print(f"Missing value count: {(monthly_commercial_summary.isna().sum()).sum()}")
print(f"Duplicated month count: {monthly_commercial_summary['order_purchase_month'].duplicated().sum()}")
print(f"whether months are in ascending order: {monthly_commercial_summary['order_purchase_month'].is_monotonic_increasing}")
print(f"Values in 2016-11 row: {monthly_commercial_summary[monthly_commercial_summary['order_purchase_month']=='2016-11']}")

monthly_commercial_summary shape: (24, 5)
Missing value count: 0
Duplicated month count: 0
whether months are in ascending order: True
Values in 2016-11 row:   order_purchase_month  monthly_order_count  monthly_item_count  monthly_gmv  \
2              2016-11                    0                   0          0.0   

   monthly_freight_value  
2                    0.0  


In [565]:
#confirm that inserting the zero month did not alter the commercial totals.
print(f"sum of monthly_order_count and delivered_orders_count are same: {np.isclose((monthly_commercial_summary['monthly_order_count'].sum()), delivered_orders_count)}")
print(f"sum of monthly_item_count and delivered_items_count are same: {np.isclose((monthly_commercial_summary['monthly_item_count'].sum()), delivered_items_count)}")
print(f"sum of monthly_gmv and GMV are same: {np.isclose((monthly_commercial_summary['monthly_gmv'].sum()), GMV)}")
print(f"sum of monthly_freight_value and total_freight_value are same: {np.isclose((monthly_commercial_summary['monthly_freight_value'].sum()), total_freight_value)}")

sum of monthly_order_count and delivered_orders_count are same: True
sum of monthly_item_count and delivered_items_count are same: True
sum of monthly_gmv and GMV are same: True
sum of monthly_freight_value and total_freight_value are same: True


In [566]:
# Calculate monthly average order value while preserving months with no orders as missing.
monthly_commercial_summary['monthly_average_order_value'] = (monthly_commercial_summary['monthly_gmv']/monthly_commercial_summary['monthly_order_count'])
monthly_commercial_summary.head()

,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value,monthly_average_order_value
0,2016-09,1,3,134.97,8.49,134.970000
1,2016-10,265,313,40325.11,6165.55,152.170226
2,2016-11,0,0,0.00,0.00,NaN
3,2016-12,1,1,10.90,8.72,10.900000
4,2017-01,750,913,111798.36,15684.01,149.064480


In [567]:
# Validate monthly average order value and its unavailable month.
print(f"monthly_average_order_value datatype: {monthly_commercial_summary['monthly_average_order_value'].dtype}")
print(f"Missing value count: {monthly_commercial_summary['monthly_average_order_value'].isna().sum()}")
print(f"Infinite value count: {(np.isinf(monthly_commercial_summary['monthly_average_order_value'])).sum()}")
missing_value_month= monthly_commercial_summary[monthly_commercial_summary['monthly_average_order_value'].isna()]
print(f"Month associated with missing value: {missing_value_month['order_purchase_month'].to_list()}")

monthly_average_order_value datatype: float64
Missing value count: 1
Infinite value count: 0
Month associated with missing value: [Period('2016-11', 'M')]


In [568]:
# Calculate monthly average items per order while preserving months with no orders as missing.
monthly_commercial_summary['monthly_average_items_per_order'] = monthly_commercial_summary['monthly_item_count']/monthly_commercial_summary['monthly_order_count']
monthly_commercial_summary.head()

,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value,monthly_average_order_value,monthly_average_items_per_order
0,2016-09,1,3,134.97,8.49,134.970000,3.000000
1,2016-10,265,313,40325.11,6165.55,152.170226,1.181132
2,2016-11,0,0,0.00,0.00,NaN,NaN
3,2016-12,1,1,10.90,8.72,10.900000,1.000000
4,2017-01,750,913,111798.36,15684.01,149.064480,1.217333


In [569]:
# Calculate monthly freight burden while preserving months with no merchandise value as missing.
monthly_commercial_summary['monthly_freight_burden_percentage'] = (monthly_commercial_summary['monthly_freight_value']/monthly_commercial_summary['monthly_gmv'])*100
monthly_commercial_summary.head()

,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value,monthly_average_order_value,monthly_average_items_per_order,monthly_freight_burden_percentage
0,2016-09,1,3,134.97,8.49,134.970000,3.000000,6.290287
1,2016-10,265,313,40325.11,6165.55,152.170226,1.181132,15.289605
2,2016-11,0,0,0.00,0.00,NaN,NaN,NaN
3,2016-12,1,1,10.90,8.72,10.900000,1.000000,80.000000
4,2017-01,750,913,111798.36,15684.01,149.064480,1.217333,14.028837


In [570]:
# Validate monthly average items and freight-burden measures.
print(f"monthly_average_items_per_order and monthly_freight_burden_percentage data type: \n{monthly_commercial_summary[['monthly_average_items_per_order', 'monthly_freight_burden_percentage']].dtypes}")
print(f"monthly_average_items_per_order and monthly_freight_burden_percentage missing value count: \n{monthly_commercial_summary[['monthly_average_items_per_order', 'monthly_freight_burden_percentage']].isna().sum()}")
print(f"monthly_average_items_per_order and monthly_freight_burden_percentage missing value count: \n{(np.isinf(monthly_commercial_summary[['monthly_average_items_per_order', 'monthly_freight_burden_percentage']])).sum()}")
missing_value_months= monthly_commercial_summary[monthly_commercial_summary['monthly_average_items_per_order'].isna() | monthly_commercial_summary['monthly_freight_burden_percentage'].isna()]
print(f"Month associated with missing value: {missing_value_months['order_purchase_month'].to_list()}")

monthly_average_items_per_order and monthly_freight_burden_percentage data type: 
monthly_average_items_per_order      float64
monthly_freight_burden_percentage    float64
dtype: object
monthly_average_items_per_order and monthly_freight_burden_percentage missing value count: 
monthly_average_items_per_order      1
monthly_freight_burden_percentage    1
dtype: int64
monthly_average_items_per_order and monthly_freight_burden_percentage missing value count: 
monthly_average_items_per_order      0
monthly_freight_burden_percentage    0
dtype: int64
Month associated with missing value: [Period('2016-11', 'M')]


### 4.2 Calculate Month-over-Month Growth

Month-over-month growth measures the percentage change in order volume, item volume, and GMV relative to the previous calendar month. Growth is unavailable for the first month and whenever the preceding month has a zero value because no valid comparison base exists.


In [571]:
# Calculate month-over-month order growth using valid previous-month order counts.
previous_month_order_count = monthly_commercial_summary['monthly_order_count'].shift(1)
previous_month_order_count = previous_month_order_count.replace(0, np.nan)

monthly_commercial_summary['monthly_order_growth_percentage'] = ((monthly_commercial_summary['monthly_order_count'] - previous_month_order_count)/previous_month_order_count)*100
monthly_commercial_summary.head()

,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value,monthly_average_order_value,monthly_average_items_per_order,monthly_freight_burden_percentage,monthly_order_growth_percentage
0,2016-09,1,3,134.97,8.49,134.970000,3.000000,6.290287,NaN
1,2016-10,265,313,40325.11,6165.55,152.170226,1.181132,15.289605,26400.0
2,2016-11,0,0,0.00,0.00,NaN,NaN,NaN,-100.0
3,2016-12,1,1,10.90,8.72,10.900000,1.000000,80.000000,NaN
4,2017-01,750,913,111798.36,15684.01,149.064480,1.217333,14.028837,74900.0


In [572]:
# Validate monthly order growth and months without valid comparisons.
print(f"monthly_order_growth_percentage data type: {monthly_commercial_summary['monthly_order_growth_percentage'].dtype}")
print(f"monthly_order_growth_percentage missing value count: {monthly_commercial_summary['monthly_order_growth_percentage'].isna().sum()}")
print(f"monthly_order_growth_percentage infinite count: {(np.isinf(monthly_commercial_summary['monthly_order_growth_percentage'])).sum()}")

monthly_order_growth_percentage_missing_value_month= monthly_commercial_summary[monthly_commercial_summary['monthly_order_growth_percentage'].isna()]
print(f"Month associated with missing value: {monthly_order_growth_percentage_missing_value_month['order_purchase_month'].to_list()}")

monthly_order_growth_percentage data type: float64
monthly_order_growth_percentage missing value count: 2
monthly_order_growth_percentage infinite count: 0
Month associated with missing value: [Period('2016-09', 'M'), Period('2016-12', 'M')]


In [573]:
# Calculate month-over-month GMV growth using valid previous-month GMV.
previous_month_gmv = monthly_commercial_summary['monthly_gmv'].shift(1)
previous_month_gmv = previous_month_gmv.replace(0, np.nan)

monthly_commercial_summary['monthly_gmv_growth_percentage'] = ((monthly_commercial_summary['monthly_gmv'] - previous_month_gmv)/previous_month_gmv)*100
monthly_commercial_summary.head()

,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value,monthly_average_order_value,monthly_average_items_per_order,monthly_freight_burden_percentage,monthly_order_growth_percentage,monthly_gmv_growth_percentage
0,2016-09,1,3,134.97,8.49,134.970000,3.000000,6.290287,NaN,NaN
1,2016-10,265,313,40325.11,6165.55,152.170226,1.181132,15.289605,26400.0,2.977709e+04
2,2016-11,0,0,0.00,0.00,NaN,NaN,NaN,-100.0,-1.000000e+02
3,2016-12,1,1,10.90,8.72,10.900000,1.000000,80.000000,NaN,NaN
4,2017-01,750,913,111798.36,15684.01,149.064480,1.217333,14.028837,74900.0,1.025573e+06


In [574]:
# Validate monthly GMV growth and months without valid comparisons.
print(f"monthly_gmv_growth_percentage data type: {monthly_commercial_summary['monthly_gmv_growth_percentage'].dtype}")
print(f"monthly_gmv_growth_percentage missing value count: {monthly_commercial_summary['monthly_gmv_growth_percentage'].isna().sum()}")
print(f"monthly_gmv_growth_percentage infinite count: {(np.isinf(monthly_commercial_summary['monthly_gmv_growth_percentage'])).sum()}")

monthly_gmv_growth_percentage_missing_value_month= monthly_commercial_summary[monthly_commercial_summary['monthly_gmv_growth_percentage'].isna()]
print(f"Month associated with missing value: {monthly_gmv_growth_percentage_missing_value_month['order_purchase_month'].to_list()}")

monthly_gmv_growth_percentage data type: float64
monthly_gmv_growth_percentage missing value count: 2
monthly_gmv_growth_percentage infinite count: 0
Month associated with missing value: [Period('2016-09', 'M'), Period('2016-12', 'M')]


### 4.3 Create the Monthly Delivery-Performance Summary

Delivery performance is summarized by purchase month at the order level. Orders without an actual customer delivery timestamp are excluded from delivery-duration averages and delivery-status rate denominators, while their missing statuses remain documented.


In [575]:
# Aggregate monthly delivery duration and delivery-status measures at the order level.
monthly_delivery_summary = delivered_orders.groupby('order_purchase_month').agg(monthly_average_delivery_duration_days= ('delivery_duration_days', 'mean'),
                                                                                monthly_valid_delivery_orders= ('delivery_status', lambda x: (x != 'missing').sum()),
                                                                                monthly_late_orders= ('delivery_status', lambda x: (x=='late').sum()),
                                                                                monthly_on_or_before_estimate_orders= ('delivery_status', lambda x: x.isin(['early', 'on_time']).sum()),
                                                                                monthly_missing_delivery_status_orders= ('delivery_status', lambda x: (x=='missing').sum())

                                                                                ).reset_index()
monthly_delivery_summary

,order_purchase_month,monthly_average_delivery_duration_days,monthly_valid_delivery_orders,monthly_late_orders,monthly_on_or_before_estimate_orders,monthly_missing_delivery_status_orders
0,2016-09,54.813194,1,1,0,0
1,2016-10,19.600559,265,2,263,0
2,2016-12,4.693021,1,0,1,0
3,2017-01,12.647044,750,22,728,0
4,2017-02,13.168825,1653,49,1604,0
5,2017-03,12.951184,2546,116,2430,0
6,2017-04,14.917913,2303,151,2152,0
7,2017-05,11.322363,3545,106,3439,1
8,2017-06,12.011573,3135,95,3040,0
9,2017-07,11.592732,3872,108,3764,0


In [576]:
# Validate monthly delivery counts against the overall delivery KPIs.
print(f"monthly_delivery_summary shape: {monthly_delivery_summary.shape}")
print(f"duplicate months: {monthly_delivery_summary['order_purchase_month'].duplicated().sum()}")
print(f"Sum of monthly_valid_delivery_orders: {monthly_delivery_summary['monthly_valid_delivery_orders'].sum()}")
print(f"Sum of monthly_late_orders: {monthly_delivery_summary['monthly_late_orders'].sum()}")
print(f"Sum of monthly_on_or_before_estimate_orders: {monthly_delivery_summary['monthly_on_or_before_estimate_orders'].sum()}")
print(f"Sum of monthly_missing_delivery_status_orders: {monthly_delivery_summary['monthly_missing_delivery_status_orders'].sum()}")

monthly_delivery_summary shape: (23, 6)
duplicate months: 0
Sum of monthly_valid_delivery_orders: 96470
Sum of monthly_late_orders: 6534
Sum of monthly_on_or_before_estimate_orders: 89936
Sum of monthly_missing_delivery_status_orders: 8


In [577]:
if (monthly_delivery_summary['monthly_valid_delivery_orders'] == (monthly_delivery_summary['monthly_late_orders'] + monthly_delivery_summary['monthly_on_or_before_estimate_orders'])).all():
    print("Monthly valid delivery orders reconcile for every month.")
else:
    print("Monthly valid delivery orders do not reconcile for every month.")

Monthly valid delivery orders reconcile for every month.


In [578]:
# Add the missing calendar month while preserving unavailable delivery averages.
monthly_delivery_summary = (monthly_delivery_summary.set_index('order_purchase_month').reindex(complete_month_range).rename_axis('order_purchase_month').reset_index())

delivery_count_columns = ['monthly_valid_delivery_orders', 'monthly_late_orders', 'monthly_on_or_before_estimate_orders', 'monthly_missing_delivery_status_orders']

monthly_delivery_summary[delivery_count_columns] = (monthly_delivery_summary[delivery_count_columns].fillna(0).astype('int'))

monthly_delivery_summary.head()

,order_purchase_month,monthly_average_delivery_duration_days,monthly_valid_delivery_orders,monthly_late_orders,monthly_on_or_before_estimate_orders,monthly_missing_delivery_status_orders
0,2016-09,54.813194,1,1,0,0
1,2016-10,19.600559,265,2,263,0
2,2016-11,NaN,0,0,0,0
3,2016-12,4.693021,1,0,1,0
4,2017-01,12.647044,750,22,728,0


In [579]:
# Calculate monthly delivery-performance rates using valid delivery orders.
monthly_delivery_summary['monthly_late_delivery_rate'] = (monthly_delivery_summary['monthly_late_orders']/monthly_delivery_summary['monthly_valid_delivery_orders'])*100
monthly_delivery_summary['monthly_on_or_before_estimate_rate'] = (monthly_delivery_summary['monthly_on_or_before_estimate_orders']/monthly_delivery_summary['monthly_valid_delivery_orders'])*100

monthly_delivery_summary.head()

,order_purchase_month,monthly_average_delivery_duration_days,monthly_valid_delivery_orders,monthly_late_orders,monthly_on_or_before_estimate_orders,monthly_missing_delivery_status_orders,monthly_late_delivery_rate,monthly_on_or_before_estimate_rate
0,2016-09,54.813194,1,1,0,0,100.000000,0.000000
1,2016-10,19.600559,265,2,263,0,0.754717,99.245283
2,2016-11,NaN,0,0,0,0,NaN,NaN
3,2016-12,4.693021,1,0,1,0,0.000000,100.000000
4,2017-01,12.647044,750,22,728,0,2.933333,97.066667


In [580]:
# Validate monthly delivery-performance rates and denominator coverage.
print(f"monthly_delivery_summary: \n{monthly_delivery_summary[['monthly_late_delivery_rate','monthly_on_or_before_estimate_rate']].dtypes}")
print(f"monthly_late_delivery_rate and monthly_on_or_before_estimate_rate missing value count: \n{monthly_delivery_summary[['monthly_late_delivery_rate', 'monthly_on_or_before_estimate_rate']].isna().sum()}")
print(f"monthly_late_delivery_rate and monthly_on_or_before_estimate_rate infinite value count: \n{(np.isinf(monthly_delivery_summary[['monthly_late_delivery_rate', 'monthly_on_or_before_estimate_rate']])).sum()}")
delivery_missing_value_months= monthly_delivery_summary[monthly_delivery_summary['monthly_late_delivery_rate'].isna() | monthly_delivery_summary['monthly_on_or_before_estimate_rate'].isna()]
print(f"Month associated with missing value: {delivery_missing_value_months['order_purchase_month'].to_list()}")

monthly_delivery_summary: 
monthly_late_delivery_rate            float64
monthly_on_or_before_estimate_rate    float64
dtype: object
monthly_late_delivery_rate and monthly_on_or_before_estimate_rate missing value count: 
monthly_late_delivery_rate            1
monthly_on_or_before_estimate_rate    1
dtype: int64
monthly_late_delivery_rate and monthly_on_or_before_estimate_rate infinite value count: 
monthly_late_delivery_rate            0
monthly_on_or_before_estimate_rate    0
dtype: int64
Month associated with missing value: [Period('2016-11', 'M')]


In [581]:
# Confirm that monthly delivery rates cover all valid delivery orders.
valid_monthly_delivery_rates = monthly_delivery_summary.dropna(subset=['monthly_late_delivery_rate', 'monthly_on_or_before_estimate_rate'])

monthly_delivery_rates_cover_all_valid_orders = np.isclose(valid_monthly_delivery_rates['monthly_late_delivery_rate']+ valid_monthly_delivery_rates['monthly_on_or_before_estimate_rate'], 100).all()
print(f"Monthly delivery rates cover all valid orders: {monthly_delivery_rates_cover_all_valid_orders}")

Monthly delivery rates cover all valid orders: True


### 4.4 Create the Monthly Customer-Review Summary

Customer-review measures are summarized by purchase month at the order level. Review averages and low-score rates include only orders with available Review records, while review coverage measures the share of all delivered orders that received a review.


In [582]:
# Aggregate monthly review coverage, scores, and response-delay measures.
monthly_review_summary = delivered_orders.groupby('order_purchase_month').agg(monthly_reviewed_orders= ('flag_missing_review_record', lambda x: (x == False).sum()),
                                                                              monthly_average_review_score= ('average_review_score', 'mean'),
                                                                              monthly_low_review_score_orders= ('average_review_score', lambda x: (x<=2).sum()),
                                                                              monthly_review_response_over_30_days_orders= ('flag_review_response_over_30_days', lambda x: (x == True).sum())
                                                                              ).reset_index()

monthly_review_summary.head()

,order_purchase_month,monthly_reviewed_orders,monthly_average_review_score,monthly_low_review_score_orders,monthly_review_response_over_30_days_orders
0,2016-09,1,1.000000,1,0
1,2016-10,262,4.011450,43,15
2,2016-12,1,5.000000,0,0
3,2017-01,741,4.199055,89,4
4,2017-02,1643,4.202678,182,17


In [583]:
# Validate monthly review measures against the overall review KPIs.
print(f"monthly_review_summary shape: {monthly_review_summary.shape}")
print(f"Duplicated months: {monthly_review_summary['order_purchase_month'].duplicated().sum()}")
print(f"Total of Montly Reviewed Orders: {monthly_review_summary['monthly_reviewed_orders'].sum()}")
print(f"Total of monthly_low_review_score_orders: {monthly_review_summary['monthly_low_review_score_orders'].sum()}")
print(f"Total of monthly_review_response_over_30_days_orders: {monthly_review_summary['monthly_review_response_over_30_days_orders'].sum()}")
print(f"Missing values in monthly_average_review_score: {monthly_review_summary['monthly_average_review_score'].isna().sum()}")

monthly_review_summary shape: (23, 5)
Duplicated months: 0
Total of Montly Reviewed Orders: 95832
Total of monthly_low_review_score_orders: 12237
Total of monthly_review_response_over_30_days_orders: 645
Missing values in monthly_average_review_score: 0


In [584]:
# Add the missing calendar month while preserving unavailable review averages.
monthly_review_summary = (monthly_review_summary.set_index('order_purchase_month').reindex(complete_month_range).rename_axis('order_purchase_month').reset_index())
review_update_0_columns = ['monthly_reviewed_orders', 'monthly_low_review_score_orders', 'monthly_review_response_over_30_days_orders']
monthly_review_summary[review_update_0_columns] = monthly_review_summary[review_update_0_columns].fillna(0).astype('int')
monthly_review_summary

,order_purchase_month,monthly_reviewed_orders,monthly_average_review_score,monthly_low_review_score_orders,monthly_review_response_over_30_days_orders
0,2016-09,1,1.000000,1,0
1,2016-10,262,4.011450,43,15
2,2016-11,0,NaN,0,0
3,2016-12,1,5.000000,0,0
4,2017-01,741,4.199055,89,4
5,2017-02,1643,4.202678,182,17
6,2017-03,2527,4.187376,284,25
7,2017-04,2290,4.137555,293,26
8,2017-05,3518,4.238488,352,35
9,2017-06,3111,4.222919,330,16


In [585]:
# Validate the continuous monthly review timeline and preserved totals.
print(f"montly_review_summary shape: {monthly_review_summary.shape}")
print(f"Duplicated month count: {monthly_review_summary['order_purchase_month'].duplicated().sum()}")
print(f"Missing count in monthly_average_review_score: {monthly_review_summary['monthly_average_review_score'].isna().sum()}")
monthly_average_review_score_missing_month = monthly_review_summary[monthly_review_summary['monthly_average_review_score'].isna()] 
print(f"Missing month: {monthly_average_review_score_missing_month['order_purchase_month']}")
print(f"Total of Montly Reviewed Orders: {monthly_review_summary['monthly_reviewed_orders'].sum()}")
print(f"Total of monthly_low_review_score_orders: {monthly_review_summary['monthly_low_review_score_orders'].sum()}")
print(f"Total of monthly_review_response_over_30_days_orders: {monthly_review_summary['monthly_review_response_over_30_days_orders'].sum()}")

montly_review_summary shape: (24, 5)
Duplicated month count: 0
Missing count in monthly_average_review_score: 1
Missing month: 2    2016-11
Name: order_purchase_month, dtype: period[M]
Total of Montly Reviewed Orders: 95832
Total of monthly_low_review_score_orders: 12237
Total of monthly_review_response_over_30_days_orders: 645


### 4.5 Combine the Monthly KPI Summaries

The monthly commercial, delivery, and review summaries share one row per calendar month. They are combined using validated one-to-one joins to create a single monthly dataset for trend analysis and reporting.


In [586]:
# Combine monthly commercial and delivery-performance measures.
monthly_kpi_summary = pd.merge(monthly_commercial_summary, monthly_delivery_summary, on='order_purchase_month', how='left', validate='one_to_one')

In [587]:
#Add monthly customer-review measures to the combined KPI summary.
monthly_kpi_summary = pd.merge(monthly_kpi_summary, monthly_review_summary, on= 'order_purchase_month', how= 'left', validate='one_to_one')

In [588]:
monthly_kpi_summary.head()

,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value,monthly_average_order_value,monthly_average_items_per_order,monthly_freight_burden_percentage,monthly_order_growth_percentage,monthly_gmv_growth_percentage,...,monthly_valid_delivery_orders,monthly_late_orders,monthly_on_or_before_estimate_orders,monthly_missing_delivery_status_orders,monthly_late_delivery_rate,monthly_on_or_before_estimate_rate,monthly_reviewed_orders,monthly_average_review_score,monthly_low_review_score_orders,monthly_review_response_over_30_days_orders
0,2016-09,1,3,134.97,8.49,134.970000,3.000000,6.290287,NaN,NaN,...,1,1,0,0,100.000000,0.000000,1,1.000000,1,0
1,2016-10,265,313,40325.11,6165.55,152.170226,1.181132,15.289605,26400.0,2.977709e+04,...,265,2,263,0,0.754717,99.245283,262,4.011450,43,15
2,2016-11,0,0,0.00,0.00,NaN,NaN,NaN,-100.0,-1.000000e+02,...,0,0,0,0,NaN,NaN,0,NaN,0,0
3,2016-12,1,1,10.90,8.72,10.900000,1.000000,80.000000,NaN,NaN,...,1,0,1,0,0.000000,100.000000,1,5.000000,0,0
4,2017-01,750,913,111798.36,15684.01,149.064480,1.217333,14.028837,74900.0,1.025573e+06,...,750,22,728,0,2.933333,97.066667,741,4.199055,89,4


In [589]:
# Validate the combined monthly KPI dataset and reconciled totals.
print(f"monthly_kpi_summary shape: {monthly_kpi_summary.shape}")
print(f"Duplicated order_purchase_month: {monthly_kpi_summary['order_purchase_month'].duplicated().sum()}")

monthly_kpi_summary shape: (24, 21)
Duplicated order_purchase_month: 0


In [590]:
print(f"Missing values by column: \n{monthly_kpi_summary.isna().sum()}")


Missing values by column: 
order_purchase_month                           0
monthly_order_count                            0
monthly_item_count                             0
monthly_gmv                                    0
monthly_freight_value                          0
monthly_average_order_value                    1
monthly_average_items_per_order                1
monthly_freight_burden_percentage              1
monthly_order_growth_percentage                2
monthly_gmv_growth_percentage                  2
monthly_average_delivery_duration_days         1
monthly_valid_delivery_orders                  0
monthly_late_orders                            0
monthly_on_or_before_estimate_orders           0
monthly_missing_delivery_status_orders         0
monthly_late_delivery_rate                     1
monthly_on_or_before_estimate_rate             1
monthly_reviewed_orders                        0
monthly_average_review_score                   1
monthly_low_review_score_orders           

In [591]:
print(f"complete 2016-11 row:")
monthly_kpi_summary[monthly_kpi_summary['order_purchase_month']=='2016-11']

complete 2016-11 row:


,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value,monthly_average_order_value,monthly_average_items_per_order,monthly_freight_burden_percentage,monthly_order_growth_percentage,monthly_gmv_growth_percentage,...,monthly_valid_delivery_orders,monthly_late_orders,monthly_on_or_before_estimate_orders,monthly_missing_delivery_status_orders,monthly_late_delivery_rate,monthly_on_or_before_estimate_rate,monthly_reviewed_orders,monthly_average_review_score,monthly_low_review_score_orders,monthly_review_response_over_30_days_orders
2,2016-11,0,0,0.0,0.0,NaN,NaN,NaN,-100.0,-100.0,...,0,0,0,0,NaN,NaN,0,NaN,0,0


In [592]:
print(f"Total of monthly_order_count: {monthly_kpi_summary['monthly_order_count'].sum()}")
print(f"Total of monthly_reviewed_orders: {monthly_kpi_summary['monthly_reviewed_orders'].sum()}")
print(f"Total of monthly_late_orders: {monthly_kpi_summary['monthly_late_orders'].sum()}")

Total of monthly_order_count: 96478
Total of monthly_reviewed_orders: 95832
Total of monthly_late_orders: 6534


In [593]:
monthly_kpi_summary.columns.tolist()

['order_purchase_month',
 'monthly_order_count',
 'monthly_item_count',
 'monthly_gmv',
 'monthly_freight_value',
 'monthly_average_order_value',
 'monthly_average_items_per_order',
 'monthly_freight_burden_percentage',
 'monthly_order_growth_percentage',
 'monthly_gmv_growth_percentage',
 'monthly_average_delivery_duration_days',
 'monthly_valid_delivery_orders',
 'monthly_late_orders',
 'monthly_on_or_before_estimate_orders',
 'monthly_missing_delivery_status_orders',
 'monthly_late_delivery_rate',
 'monthly_on_or_before_estimate_rate',
 'monthly_reviewed_orders',
 'monthly_average_review_score',
 'monthly_low_review_score_orders',
 'monthly_review_response_over_30_days_orders']

In [594]:
# Calculate monthly review coverage, low-score, and response-delay rates.
monthly_kpi_summary['monthly_review_coverage_rate'] = (monthly_kpi_summary['monthly_reviewed_orders']/monthly_kpi_summary['monthly_order_count'])*100
monthly_kpi_summary['monthly_low_review_score_rate'] = (monthly_kpi_summary['monthly_low_review_score_orders']/monthly_kpi_summary['monthly_reviewed_orders'])*100
monthly_kpi_summary['monthly_review_response_over_30_days_rate'] = (monthly_kpi_summary['monthly_review_response_over_30_days_orders']/monthly_kpi_summary['monthly_reviewed_orders'])*100

In [595]:
monthly_kpi_summary.head()

,order_purchase_month,monthly_order_count,monthly_item_count,monthly_gmv,monthly_freight_value,monthly_average_order_value,monthly_average_items_per_order,monthly_freight_burden_percentage,monthly_order_growth_percentage,monthly_gmv_growth_percentage,...,monthly_missing_delivery_status_orders,monthly_late_delivery_rate,monthly_on_or_before_estimate_rate,monthly_reviewed_orders,monthly_average_review_score,monthly_low_review_score_orders,monthly_review_response_over_30_days_orders,monthly_review_coverage_rate,monthly_low_review_score_rate,monthly_review_response_over_30_days_rate
0,2016-09,1,3,134.97,8.49,134.970000,3.000000,6.290287,NaN,NaN,...,0,100.000000,0.000000,1,1.000000,1,0,100.000000,100.000000,0.000000
1,2016-10,265,313,40325.11,6165.55,152.170226,1.181132,15.289605,26400.0,2.977709e+04,...,0,0.754717,99.245283,262,4.011450,43,15,98.867925,16.412214,5.725191
2,2016-11,0,0,0.00,0.00,NaN,NaN,NaN,-100.0,-1.000000e+02,...,0,NaN,NaN,0,NaN,0,0,NaN,NaN,NaN
3,2016-12,1,1,10.90,8.72,10.900000,1.000000,80.000000,NaN,NaN,...,0,0.000000,100.000000,1,5.000000,0,0,100.000000,0.000000,0.000000
4,2017-01,750,913,111798.36,15684.01,149.064480,1.217333,14.028837,74900.0,1.025573e+06,...,0,2.933333,97.066667,741,4.199055,89,4,98.800000,12.010796,0.539811


In [596]:
# Validate monthly review rates and unavailable denominators.
monthly_review_rate_columns = ['monthly_review_coverage_rate', 'monthly_low_review_score_rate', 'monthly_review_response_over_30_days_rate']
print(f"monthly_review_rate_columns data types: \n{monthly_kpi_summary[monthly_review_rate_columns].dtypes}")
print("\n")
print(f"monthly_review_rate_columns missing value count: \n{monthly_kpi_summary[monthly_review_rate_columns].isna().sum()}")
print("\n")
print(f"monthly_review_rate_columns infinite value count: \n{(np.isinf(monthly_kpi_summary[monthly_review_rate_columns])).sum()}")
print("\n")
monthly_review_rate_columns_missing_month = monthly_kpi_summary[monthly_kpi_summary['monthly_review_coverage_rate'].isna() | monthly_kpi_summary['monthly_low_review_score_rate'].isna() | monthly_kpi_summary['monthly_review_response_over_30_days_rate'].isna()]
print(f"Missing month: {monthly_review_rate_columns_missing_month['order_purchase_month']}")

monthly_review_rate_columns data types: 
monthly_review_coverage_rate                 float64
monthly_low_review_score_rate                float64
monthly_review_response_over_30_days_rate    float64
dtype: object


monthly_review_rate_columns missing value count: 
monthly_review_coverage_rate                 1
monthly_low_review_score_rate                1
monthly_review_response_over_30_days_rate    1
dtype: int64


monthly_review_rate_columns infinite value count: 
monthly_review_coverage_rate                 0
monthly_low_review_score_rate                0
monthly_review_response_over_30_days_rate    0
dtype: int64


Missing month: 2    2016-11
Name: order_purchase_month, dtype: period[M]


In [597]:
print(f"monthly_review_coverage_rate"+"\n"
      f"minimum value: {monthly_kpi_summary['monthly_review_coverage_rate'].min()}"+"\n"+f"maximum value: {monthly_kpi_summary['monthly_review_coverage_rate'].max()}")
print("\n")
print(f"monthly_low_review_score_rate"+"\n"
      f"minimum value: {monthly_kpi_summary['monthly_low_review_score_rate'].min()}"+"\n"+f"maximum value: {monthly_kpi_summary['monthly_low_review_score_rate'].max()}")
print("\n")
print(f"monthly_review_response_over_30_days_rate"+"\n"
      f"minimum value: {monthly_kpi_summary['monthly_review_response_over_30_days_rate'].min()}"+"\n"+f"maximum value: {monthly_kpi_summary['monthly_review_response_over_30_days_rate'].max()}")


monthly_review_coverage_rate
minimum value: 98.8
maximum value: 100.0


monthly_low_review_score_rate
minimum value: 0.0
maximum value: 100.0


monthly_review_response_over_30_days_rate
minimum value: 0.0
maximum value: 5.7251908396946565


In [598]:
# Validate the completed monthly KPI summary before trend analysis.
print(f"monthly_kpi_summary shape: {monthly_kpi_summary.shape}")
print(f"duplicated months: {monthly_kpi_summary['order_purchase_month'].duplicated().sum()}")
print(f"Chronological month order: {monthly_kpi_summary['order_purchase_month'].is_monotonic_increasing}")
print(f"Infinite values across numeric columns: {((np.isinf(monthly_kpi_summary.select_dtypes(include='number'))).sum()).sum()}")

monthly_kpi_summary shape: (24, 24)
duplicated months: 0
Chronological month order: True
Infinite values across numeric columns: 0


In [599]:
print(f"Missing values by column: \n{monthly_kpi_summary.isna().sum()}")

Missing values by column: 
order_purchase_month                           0
monthly_order_count                            0
monthly_item_count                             0
monthly_gmv                                    0
monthly_freight_value                          0
monthly_average_order_value                    1
monthly_average_items_per_order                1
monthly_freight_burden_percentage              1
monthly_order_growth_percentage                2
monthly_gmv_growth_percentage                  2
monthly_average_delivery_duration_days         1
monthly_valid_delivery_orders                  0
monthly_late_orders                            0
monthly_on_or_before_estimate_orders           0
monthly_missing_delivery_status_orders         0
monthly_late_delivery_rate                     1
monthly_on_or_before_estimate_rate             1
monthly_reviewed_orders                        0
monthly_average_review_score                   1
monthly_low_review_score_orders           

### 4.6 Define the Comparable Monthly Trend Period

The complete monthly dataset is retained for transparency. However, the 2016 months contain extremely low or zero order activity, producing unstable percentage changes that are not representative of established business performance. Comparative monthly rankings therefore use January 2017 through August 2018, while the full 24-month table remains available for historical reference.


In [600]:
# Create the comparable monthly KPI period for reliable trend analysis.
comparable_monthly_kpis = monthly_kpi_summary[monthly_kpi_summary['order_purchase_month']>='2017-01'].copy()
comparable_monthly_kpis = comparable_monthly_kpis.reset_index(drop=True)

In [601]:
comparable_monthly_kpis.shape

(20, 24)

In [602]:
# Validate the comparable monthly KPI period before trend comparisons.
print(f"Earliest month: {comparable_monthly_kpis['order_purchase_month'].min()}")
print(f"Latest month: {comparable_monthly_kpis['order_purchase_month'].max()}")
print(f"Duplicated month value count: {comparable_monthly_kpis['order_purchase_month'].duplicated().sum()}")
print(f"Months remain chronologically sorted: {comparable_monthly_kpis['order_purchase_month'].is_monotonic_increasing}")
print(f"Missing values across the subset: {(comparable_monthly_kpis.isna().sum()).sum()}")


Earliest month: 2017-01
Latest month: 2018-08
Duplicated month value count: 0
Months remain chronologically sorted: True
Missing values across the subset: 0


## 5. Analyze Category and Product Performance

Category and product performance is calculated from the delivered-order-item dataset because each row represents one purchased item. Item counts, merchandise value, freight value, and contribution percentages are summarized without repeating order-level measures. Records without an available English category name are retained as a separate missing-category group.

### 5.1 Validate the Category Analysis Scope


In [603]:
# Inspect missing and available category information before aggregation.
delivered_missing_product_name_english = delivered_order_items[delivered_order_items['product_category_name_english'].isna()]

print(f"Missing items row count: {len(delivered_missing_product_name_english)}")
print(f"Distinct products among missing-category rows: {delivered_missing_product_name_english['product_id'].nunique()}")
print(f"Distinct orders among missing-category rows: {delivered_missing_product_name_english['order_id'].nunique()}")
print(f"Number of available category names: {delivered_order_items['product_category_name_english'].nunique()}")

Missing items row count: 1559
Distinct products among missing-category rows: 595
Distinct orders among missing-category rows: 1412
Number of available category names: 71


In [604]:
# Create the category-analysis scope and retain unavailable categories as a separate group.
category_analysis_items = delivered_order_items.copy()

In [605]:
category_analysis_items['product_category_name_english'] = category_analysis_items['product_category_name_english'].fillna("missing")

In [606]:
print(f"product_category_name_english missing value count: {category_analysis_items['product_category_name_english'].isna().sum()}")
print(f"Unique category label count : {category_analysis_items['product_category_name_english'].nunique()}")
print(f"Row labeled missing: {len(category_analysis_items[category_analysis_items['product_category_name_english']=="missing"])} ")

product_category_name_english missing value count: 0
Unique category label count : 72
Row labeled missing: 1559 


### 5.2 Create the Category-Performance Table

Delivered item records are grouped by English product category to measure sales volume, order reach, product variety, merchandise value, and freight value. The separate `missing` category preserves items whose category information is unavailable.


In [607]:
# Aggregate delivered item, order, product, merchandise, and freight measures by category.
category_performance = category_analysis_items.groupby('product_category_name_english').agg(category_item_count= ('order_item_id', 'count'),
                                                                                            category_order_count= ('order_id', 'nunique'),
                                                                                            category_product_count= ('product_id', 'nunique'),
                                                                                            category_gmv= ('price', 'sum'),
                                                                                            category_freight_value= ('freight_value', 'sum')
                                                                                            ).reset_index()
category_performance

,product_category_name_english,category_item_count,category_order_count,category_product_count,category_gmv,category_freight_value
0,agro_industry_and_commerce,206,177,71,70566.10,5637.20
1,air_conditioning,289,246,121,53323.56,6530.99
2,art,197,195,52,23657.30,3828.23
3,arts_and_craftmanship,24,23,19,1814.01,370.13
4,audio,362,348,58,50570.60,5684.72
...,...,...,...,...,...,...
67,stationery,2466,2264,831,223788.69,45786.36
68,tablets_printing_image,83,79,9,7528.41,1226.20
69,telephony,4430,4093,1112,309860.23,69342.39
70,toys,4030,3804,1373,471286.48,75774.58


In [608]:
# Validate the category-performance table against delivered-item totals.
print(f"category_performance shape: {category_performance.shape}")
print(f"Duplicated category name: {category_performance['product_category_name_english'].duplicated().sum()}")
print(f"Missing values across the table: {(category_performance.isna().sum()).sum()}")
print(f"Sum of category_item_count: {category_performance['category_item_count'].sum()}")
print(f"Sum of category_gmv: {category_performance['category_gmv'].sum()}")
print(f"Sum of category_freight_value: {category_performance['category_freight_value'].sum()}")

category_performance shape: (72, 6)
Duplicated category name: 0
Missing values across the table: 0
Sum of category_item_count: 110197
Sum of category_gmv: 13221498.11
Sum of category_freight_value: 2198275.64


In [609]:
# Calculate each category’s share of delivered items.
category_performance['category_item_share_percentage'] = (category_performance['category_item_count']/delivered_items_count)*100
category_performance

,product_category_name_english,category_item_count,category_order_count,category_product_count,category_gmv,category_freight_value,category_item_share_percentage
0,agro_industry_and_commerce,206,177,71,70566.10,5637.20,0.186938
1,air_conditioning,289,246,121,53323.56,6530.99,0.262258
2,art,197,195,52,23657.30,3828.23,0.178771
3,arts_and_craftmanship,24,23,19,1814.01,370.13,0.021779
4,audio,362,348,58,50570.60,5684.72,0.328503
...,...,...,...,...,...,...,...
67,stationery,2466,2264,831,223788.69,45786.36,2.237810
68,tablets_printing_image,83,79,9,7528.41,1226.20,0.075320
69,telephony,4430,4093,1112,309860.23,69342.39,4.020073
70,toys,4030,3804,1373,471286.48,75774.58,3.657087


In [610]:
# Confirm that category item shares account for all delivered items.
print(f"Category shares collectively equal approximately 100%: {np.isclose(category_performance['category_item_share_percentage'].sum(), 100)}")

Category shares collectively equal approximately 100%: True


In [611]:
# Calculate each category’s share of total delivered merchandise value.
category_performance['category_gmv_share_percentage'] = (category_performance['category_gmv']/GMV)*100
category_performance['category_gmv_share_percentage'] 

0     0.533722
1     0.403310
2     0.178931
3     0.013720
4     0.382488
        ...   
67    1.692612
68    0.056941
69    2.343609
70    3.564547
71    8.820309
Name: category_gmv_share_percentage, Length: 72, dtype: float64

In [612]:
# Confirm that category GMV shares account for total delivered GMV.
print(f"Category GMV shares collectively equal approximately 100%: {np.isclose(category_performance['category_gmv_share_percentage'].sum(), 100)}")

Category GMV shares collectively equal approximately 100%: True


In [613]:
# Calculate the average merchandise price per delivered item in each category.
category_performance['category_average_item_price'] = category_performance['category_gmv']/category_performance['category_item_count']
category_performance['category_average_item_price']

0     342.553883
1     184.510588
2     120.087817
3      75.583750
4     139.697790
         ...    
67     90.749672
68     90.703735
69     69.945876
70    116.944536
71    199.040276
Name: category_average_item_price, Length: 72, dtype: float64

In [614]:
# Calculate freight cost as a percentage of merchandise value within each category.
category_performance['category_freight_burden_percentage'] = (category_performance['category_freight_value']/category_performance['category_gmv'])*100
category_performance['category_freight_burden_percentage']

0      7.988538
1     12.247851
2     16.182024
3     20.403967
4     11.241156
        ...    
67    20.459640
68    16.287636
69    22.378603
70    16.078242
71     8.416916
Name: category_freight_burden_percentage, Length: 72, dtype: float64

In [615]:
# Validate the derived category-performance measures.
delivery_category_columns = ['category_item_share_percentage', 'category_gmv_share_percentage', 'category_average_item_price', 'category_freight_burden_percentage']

print(f"delivered_category_columns data type: \n{category_performance[delivery_category_columns].dtypes}")
print("\n")
print(f"Missing value counts: \n{category_performance[delivery_category_columns].isna().sum()}")
print("\n")
print(f"Infinite value counts: \n{(np.isinf(category_performance[delivery_category_columns])).sum()}")


delivered_category_columns data type: 
category_item_share_percentage        float64
category_gmv_share_percentage         float64
category_average_item_price           float64
category_freight_burden_percentage    float64
dtype: object


Missing value counts: 
category_item_share_percentage        0
category_gmv_share_percentage         0
category_average_item_price           0
category_freight_burden_percentage    0
dtype: int64


Infinite value counts: 
category_item_share_percentage        0
category_gmv_share_percentage         0
category_average_item_price           0
category_freight_burden_percentage    0
dtype: int64


In [616]:
print(f"category_item_share_percentage:"+"\n"
      f"Minimum value: {category_performance['category_item_share_percentage'].min()}"+"\n" + f"Maximum value: {category_performance['category_item_share_percentage'].max()}")
print("\n")
print(f"category_gmv_share_percentage:"+"\n"
      f"Minimum value: {category_performance['category_gmv_share_percentage'].min()}"+"\n" + f"Maximum value: {category_performance['category_gmv_share_percentage'].max()}")
print("\n")
print(f"category_average_item_price:"+"\n"
      f"Minimum value: {category_performance['category_average_item_price'].min()}"+"\n" + f"Maximum value: {category_performance['category_average_item_price'].max()}")
print("\n")
print(f"category_freight_burden_percentage:"+"\n"
      f"Minimum value: {category_performance['category_freight_burden_percentage'].min()}"+"\n" + f"Maximum value: {category_performance['category_freight_burden_percentage'].max()}")

category_item_share_percentage:
Minimum value: 0.0018149314409648177
Maximum value: 9.939472036443824


category_gmv_share_percentage:
Minimum value: 0.0021426467533640177
Maximum value: 9.32671706141476


category_average_item_price:
Minimum value: 25.342333333333332
Maximum value: 1098.9152763819095


category_freight_burden_percentage:
Minimum value: 4.419899861050737
Maximum value: 53.968984702802956


#### Category-Performance Table Validation Summary

The category-performance table contains 72 category groups, including a separate group for unavailable category information. Item counts, merchandise value, and freight value reconcile fully with the delivered-item dataset. Category item shares and GMV shares each total approximately 100%, while all derived measures contain no missing or infinite values. The table is therefore suitable for category ranking and low-performance screening.

### 5.3 Evaluate Category Performance and Identify Low-Performance Candidates

Available categories are evaluated using delivered-item volume, GMV contribution, product variety, average item price, and freight burden. Category item-count and GMV distributions are examined before defining low performance so that the screening thresholds are based on the observed data rather than arbitrary cutoffs. The unavailable-category group is retained in the complete category table but excluded from business rankings and performance classification.

In [617]:
# Rank available categories by delivered item count.
category_items_count_mask = category_performance['product_category_name_english'] != 'missing'
category_item_count_ranking = category_performance.loc[category_items_count_mask, 'category_item_count'].rank(method='dense', ascending=False)

category_performance['category_item_count_rank'] =category_item_count_ranking

In [618]:
category_performance

,product_category_name_english,category_item_count,category_order_count,category_product_count,category_gmv,category_freight_value,category_item_share_percentage,category_gmv_share_percentage,category_average_item_price,category_freight_burden_percentage,category_item_count_rank
0,agro_industry_and_commerce,206,177,71,70566.10,5637.20,0.186938,0.533722,342.553883,7.988538,44.0
1,air_conditioning,289,246,121,53323.56,6530.99,0.262258,0.403310,184.510588,12.247851,35.0
2,art,197,195,52,23657.30,3828.23,0.178771,0.178931,120.087817,16.182024,46.0
3,arts_and_craftmanship,24,23,19,1814.01,370.13,0.021779,0.013720,75.583750,20.403967,64.0
4,audio,362,348,58,50570.60,5684.72,0.328503,0.382488,139.697790,11.241156,31.0
...,...,...,...,...,...,...,...,...,...,...,...
67,stationery,2466,2264,831,223788.69,45786.36,2.237810,1.692612,90.749672,20.459640,16.0
68,tablets_printing_image,83,79,9,7528.41,1226.20,0.075320,0.056941,90.703735,16.287636,52.0
69,telephony,4430,4093,1112,309860.23,69342.39,4.020073,2.343609,69.945876,22.378603,8.0
70,toys,4030,3804,1373,471286.48,75774.58,3.657087,3.564547,116.944536,16.078242,11.0


In [619]:
# Validate the category item-count ranking and confirm the missing-category group remains unranked.
missing_category_rank_validation = category_performance[category_performance['product_category_name_english']=='missing']
missing_category_rank_validation[['product_category_name_english', 'category_item_count_rank']]

,product_category_name_english,category_item_count_rank
55,missing,NaN


In [620]:
print(f"Minimum category rank: {category_performance['category_item_count_rank'].min()}")
print(f"Missing category rank: {category_performance['category_item_count_rank'].isna().sum()}")
print(f"Maximum category rank: {category_performance['category_item_count_rank'].max()}")

Minimum category rank: 1.0
Missing category rank: 1
Maximum category rank: 67.0


In [621]:
# Inspect the highest- and lowest-selling available categories by delivered item count.
highest_selling_categories = category_performance[category_performance['category_item_count_rank'] == (category_performance['category_item_count_rank'].min())]
lowest_selling_categories = category_performance[category_performance['category_item_count_rank'] == (category_performance['category_item_count_rank'].max())]

# Examine the distribution of category item counts before defining high-performing categories.
highest_selling_categories[['product_category_name_english', 'category_item_count', 'category_item_share_percentage', 'category_gmv', 'category_gmv_share_percentage', 'category_item_count_rank']]


,product_category_name_english,category_item_count,category_item_share_percentage,category_gmv,category_gmv_share_percentage,category_item_count_rank
7,bed_bath_table,10953,9.939472,1023434.76,7.740687,1.0


In [622]:
# Examine the distribution of category item counts before defining low-performing categories.
lowest_selling_categories[['product_category_name_english', 'category_item_count', 'category_item_share_percentage', 'category_gmv', 'category_gmv_share_percentage', 'category_item_count_rank']]

,product_category_name_english,category_item_count,category_item_share_percentage,category_gmv,category_gmv_share_percentage,category_item_count_rank
62,security_and_services,2,0.001815,283.29,0.002143,67.0


In [623]:
# Examine the delivered item-count distribution across available categories.
category_performance_without_missing_category_name = category_performance[category_performance['product_category_name_english']!= 'missing']
category_performance_without_missing_category_name['category_item_count'].describe()

count       71.000000
mean      1530.112676
std       2553.036056
min          2.000000
25%         93.000000
50%        274.000000
75%       1796.000000
max      10953.000000
Name: category_item_count, dtype: float64

In [624]:
# Flag available categories in the bottom item-count quartile as low-volume candidates.
category_performance['low_volume_candidate'] =(category_performance['product_category_name_english']!='missing') & (category_performance['category_item_count'] <= (category_performance_without_missing_category_name['category_item_count'].quantile(0.25)))
category_performance['low_volume_candidate'].sum()

np.int64(18)

In [625]:
# Confirm that the unavailable-category group is excluded from low-volume classification.
((category_performance['product_category_name_english'] =='missing') & (category_performance['low_volume_candidate'] == False)).sum()

np.int64(1)

In [626]:
# Inspect low-volume category candidates using commercial and product-range measures.
low_volume_category_candidates = category_performance[category_performance['low_volume_candidate']==True].sort_values(by='category_item_count', ascending=True)
low_volume_category_candidates[['product_category_name_english', 'category_item_count', 'category_item_share_percentage', 'category_product_count', 'category_gmv', 'category_gmv_share_percentage', 'category_average_item_price', 'category_freight_burden_percentage']]


,product_category_name_english,category_item_count,category_item_share_percentage,category_product_count,category_gmv,category_gmv_share_percentage,category_average_item_price,category_freight_burden_percentage
62,security_and_services,2,0.001815,2,283.29,0.002143,141.645000,14.550461
29,fashion_childrens_clothes,7,0.006352,4,519.95,0.003933,74.278571,15.139917
52,la_cuisine,14,0.012705,10,2054.99,0.015543,146.785000,16.231223
11,cds_dvds_musicals,14,0.012705,1,730.00,0.005521,52.142857,30.820548
3,arts_and_craftmanship,24,0.021779,19,1814.01,0.013720,75.583750,20.403967
32,fashion_sport,29,0.026317,19,2094.52,0.015842,72.224828,26.881099
46,home_comfort_2,30,0.027224,5,760.27,0.005750,25.342333,53.968985
35,flowers,33,0.029946,14,1110.04,0.008396,33.637576,44.040755
23,diapers_and_hygiene,37,0.033576,11,1500.79,0.011351,40.561892,36.340860
41,furniture_mattress_and_upholstery,37,0.033576,10,4323.38,0.032700,116.848108,36.577169


In [627]:
# Validate the low-volume category candidate subset.
print(f"low_volume_category_candidates shape: {low_volume_category_candidates.shape}")
print(f"Every low_volume_candidate value: {low_volume_category_candidates['low_volume_candidate'].all()}")
print(f"Maximum category_item_count: {low_volume_category_candidates['category_item_count'].max()}")
print(f"Missing category count: {(low_volume_category_candidates['product_category_name_english']=='missing').sum()}")

low_volume_category_candidates shape: (18, 12)
Every low_volume_candidate value: True
Maximum category_item_count: 83
Missing category count: 0


In [628]:
# Flag available categories in the bottom GMV quartile as low-GMV candidates.
category_performance['low_gmv_candidate'] = (category_performance['product_category_name_english']!='missing') & (category_performance['category_gmv'] <= (category_performance_without_missing_category_name['category_gmv'].quantile(0.25)))

In [629]:
# validate low_gmv_candidates
print(f"Number of low_gmv_candidates True values:",  {category_performance['low_gmv_candidate'].sum()})
print("The GMV 25th-percentile threshold:" , (category_performance_without_missing_category_name['category_gmv'].quantile(0.25)))
print("Missing category flag:", category_performance.loc[category_performance['product_category_name_english'] == 'missing','low_gmv_candidate'].iloc[0])

Number of low_gmv_candidates True values: {np.int64(18)}
The GMV 25th-percentile threshold: 9021.895
Missing category flag: False


In [630]:
# Flag categories that fall in the bottom quartile for both item volume and GMV.
category_performance['low_performance_candidate'] = (category_performance['low_volume_candidate'] == True) & (category_performance['low_gmv_candidate'] == True)

In [631]:
# validate low_performance_candidate 
print("Number of categories flagged:", category_performance['low_performance_candidate'].sum())
print("missing category value:",  category_performance.loc[category_performance['product_category_name_english'] == 'missing','low_performance_candidate'].iloc[0])
candidates_with_both_flag_true = category_performance[category_performance['low_performance_candidate'] == True]
print("Every final candidate must have both component flags equal to True:",((candidates_with_both_flag_true[['low_volume_candidate', 'low_gmv_candidate']]).all()).all())

Number of categories flagged: 17
missing category value: False
Every final candidate must have both component flags equal to True: True


In [632]:
# Create the final category-level low-performance candidate table.
low_performance_category_candidates = category_performance[category_performance['low_performance_candidate'] == True].sort_values(by=['category_item_count', 'category_gmv'], ascending=True)
low_performance_category_candidates[['product_category_name_english', 'category_item_count', 'category_item_share_percentage', 'category_order_count', 'category_product_count', 'category_gmv', 'category_gmv_share_percentage', 'category_average_item_price', 'category_freight_burden_percentage']]

,product_category_name_english,category_item_count,category_item_share_percentage,category_order_count,category_product_count,category_gmv,category_gmv_share_percentage,category_average_item_price,category_freight_burden_percentage
62,security_and_services,2,0.001815,2,2,283.29,0.002143,141.645000,14.550461
29,fashion_childrens_clothes,7,0.006352,7,4,519.95,0.003933,74.278571,15.139917
11,cds_dvds_musicals,14,0.012705,12,1,730.00,0.005521,52.142857,30.820548
52,la_cuisine,14,0.012705,13,10,2054.99,0.015543,146.785000,16.231223
3,arts_and_craftmanship,24,0.021779,23,19,1814.01,0.013720,75.583750,20.403967
32,fashion_sport,29,0.026317,26,19,2094.52,0.015842,72.224828,26.881099
46,home_comfort_2,30,0.027224,24,5,760.27,0.005750,25.342333,53.968985
35,flowers,33,0.029946,29,14,1110.04,0.008396,33.637576,44.040755
23,diapers_and_hygiene,37,0.033576,25,11,1500.79,0.011351,40.561892,36.340860
41,furniture_mattress_and_upholstery,37,0.033576,37,10,4323.38,0.032700,116.848108,36.577169


In [633]:
# Validate the final category-level low-performance candidate table.
print("Row count:", len(low_performance_category_candidates))
print("Duplicate category names:", low_performance_category_candidates['product_category_name_english'].duplicated().sum())
print("Missing value count:", low_performance_category_candidates.isna().sum().sum())
print("Missing category count:", (low_performance_category_candidates['product_category_name_english'] == 'missing').sum())

Row count: 17
Duplicate category names: 0
Missing value count: 0
Missing category count: 0


In [634]:
print("All low_performance_candidate values are True:", low_performance_category_candidates['low_performance_candidate'].all())
print("Maximum item count is less than or equal to the available-category item-count 25th percentile:", ((low_performance_category_candidates['category_item_count'].max())<=(category_performance_without_missing_category_name['category_item_count'].quantile(0.25))))
print("Maximum GMV is less than or equal to the available-category GMV 25th percentile:", ((low_performance_category_candidates['category_gmv'].max())<=(category_performance_without_missing_category_name['category_gmv'].quantile(0.25))))


All low_performance_candidate values are True: True
Maximum item count is less than or equal to the available-category item-count 25th percentile: True
Maximum GMV is less than or equal to the available-category GMV 25th percentile: True


#### Low-Performance Category Screening Summary

Available categories were screened independently using delivered-item volume and GMV. Categories at or below the 25th percentile for both measures were classified as low-performance candidates. This produced 17 categories after excluding records with unavailable category information. The classification is an analytical screening rule rather than definitive proof of poor performance because category maturity, product availability, profitability, and strategic importance are unavailable in the dataset.

### 5.4 Create the Product-Performance Table

Product performance is evaluated at the product-within-category level using delivered-item volume, order reach, merchandise value, freight value, and contribution to the corresponding category. Products with unavailable category information are retained for reconciliation but excluded from category-based business rankings and low-performance classification.

In [635]:
# Validate product identifiers and product-to-category consistency before aggregation.
print("Missing product_id count: ", category_analysis_items['product_id'].isna().sum())
print("Number of distinct delivered products: ", category_analysis_items['product_id'].nunique())
print("Distinct products in the Missing category: ", delivered_missing_product_name_english['product_id'].nunique())
print("Whether any product_id maps to more than one category label: ", (category_analysis_items.groupby('product_id')['product_category_name_english'].nunique()>1).sum())

Missing product_id count:  0
Number of distinct delivered products:  32216
Distinct products in the Missing category:  595
Whether any product_id maps to more than one category label:  0


#### 5.4.1 Validate the Product-Analysis Scope

Product identifiers and category assignments are validated before aggregation to ensure that the product-performance table has a reliable product-within-category grain. The checks confirm that delivered product records contain valid product identifiers and that each product maps consistently to one category label.

In [636]:
# Aggregate delivered item, order, merchandise, and freight measures by product within category.
product_performance  = category_analysis_items.groupby(['product_category_name_english', 'product_id']).agg(product_item_count= ('order_item_id', 'count'),
                                                                                                            product_order_count= ('order_id', 'nunique'),
                                                                                                            product_gmv= ('price', 'sum'),
                                                                                                            product_freight_value= ('freight_value', 'sum')
                                                                                                            ).reset_index()
product_performance.head()

,product_category_name_english,product_id,product_item_count,product_order_count,product_gmv,product_freight_value
0,agro_industry_and_commerce,018ca97302e4293050cc41413194bb19,1,1,88.00,11.95
1,agro_industry_and_commerce,026f43af35e7951067097527d5c31bcc,2,1,359.80,62.68
2,agro_industry_and_commerce,07f01b6fcacc1b187a71e5074199db2d,1,1,57.00,29.99
3,agro_industry_and_commerce,0a0adf0de1769b2970f3096d0b5abfb7,1,1,58.50,29.11
4,agro_industry_and_commerce,0a27862bbf658a5b8cf655761bc5c72d,1,1,51.73,66.89


In [637]:
# Validate the product-performance table against delivered-item totals.
print("product_performance shape: ", product_performance.shape)
print("Duplicate combinations of category and product_id: ",product_performance[['product_category_name_english', 'product_id']].duplicated().sum())
print("Duplicate product_id values: ", product_performance['product_id'].duplicated().sum())
print("Missing values across the table: ", product_performance.isna().sum().sum())
print("Sum of product_item_count: ", product_performance['product_item_count'].sum())
print("Sum of product_gmv: ", product_performance['product_gmv'].sum())
print("Sum of product_freight_value: ", product_performance['product_freight_value'].sum())

product_performance shape:  (32216, 6)
Duplicate combinations of category and product_id:  0
Duplicate product_id values:  0
Missing values across the table:  0
Sum of product_item_count:  110197
Sum of product_gmv:  13221498.11
Sum of product_freight_value:  2198275.6399999997


In [638]:
# Calculate each product’s delivered-item contribution within its category.
product_performance['product_item_contribution_within_category_percentage'] = (product_performance['product_item_count']/product_performance.groupby('product_category_name_english')['product_item_count'].transform('sum'))*100
product_performance.head()

,product_category_name_english,product_id,product_item_count,product_order_count,product_gmv,product_freight_value,product_item_contribution_within_category_percentage
0,agro_industry_and_commerce,018ca97302e4293050cc41413194bb19,1,1,88.00,11.95,0.485437
1,agro_industry_and_commerce,026f43af35e7951067097527d5c31bcc,2,1,359.80,62.68,0.970874
2,agro_industry_and_commerce,07f01b6fcacc1b187a71e5074199db2d,1,1,57.00,29.99,0.485437
3,agro_industry_and_commerce,0a0adf0de1769b2970f3096d0b5abfb7,1,1,58.50,29.11,0.485437
4,agro_industry_and_commerce,0a27862bbf658a5b8cf655761bc5c72d,1,1,51.73,66.89,0.485437


In [639]:
# Confirm that product item contributions sum to approximately 100% within every category.
print((np.isclose((product_performance.groupby('product_category_name_english')['product_item_contribution_within_category_percentage'].sum()), 100)).all())

True


In [640]:
# Calculate each product’s GMV contribution within its category.
product_performance['product_gmv_contribution_within_category_percentage'] = (product_performance['product_gmv']/product_performance.groupby('product_category_name_english')['product_gmv'].transform('sum'))*100
product_performance.head()

,product_category_name_english,product_id,product_item_count,product_order_count,product_gmv,product_freight_value,product_item_contribution_within_category_percentage,product_gmv_contribution_within_category_percentage
0,agro_industry_and_commerce,018ca97302e4293050cc41413194bb19,1,1,88.00,11.95,0.485437,0.124706
1,agro_industry_and_commerce,026f43af35e7951067097527d5c31bcc,2,1,359.80,62.68,0.970874,0.509877
2,agro_industry_and_commerce,07f01b6fcacc1b187a71e5074199db2d,1,1,57.00,29.99,0.485437,0.080775
3,agro_industry_and_commerce,0a0adf0de1769b2970f3096d0b5abfb7,1,1,58.50,29.11,0.485437,0.082901
4,agro_industry_and_commerce,0a27862bbf658a5b8cf655761bc5c72d,1,1,51.73,66.89,0.485437,0.073307


In [641]:
# Confirm that product GMV contributions sum to approximately 100% within every category.
print((np.isclose((product_performance.groupby('product_category_name_english')['product_gmv_contribution_within_category_percentage'].sum()), 100)).all())

True


In [642]:
# Calculate average item price and freight burden for each product.
product_performance['product_average_item_price'] = product_performance['product_gmv']/product_performance['product_item_count']
product_performance['product_freight_burden_percentage'] = (product_performance['product_freight_value']/product_performance['product_gmv'])*100
product_performance.head()

,product_category_name_english,product_id,product_item_count,product_order_count,product_gmv,product_freight_value,product_item_contribution_within_category_percentage,product_gmv_contribution_within_category_percentage,product_average_item_price,product_freight_burden_percentage
0,agro_industry_and_commerce,018ca97302e4293050cc41413194bb19,1,1,88.00,11.95,0.485437,0.124706,88.00,13.579545
1,agro_industry_and_commerce,026f43af35e7951067097527d5c31bcc,2,1,359.80,62.68,0.970874,0.509877,179.90,17.420789
2,agro_industry_and_commerce,07f01b6fcacc1b187a71e5074199db2d,1,1,57.00,29.99,0.485437,0.080775,57.00,52.614035
3,agro_industry_and_commerce,0a0adf0de1769b2970f3096d0b5abfb7,1,1,58.50,29.11,0.485437,0.082901,58.50,49.760684
4,agro_industry_and_commerce,0a27862bbf658a5b8cf655761bc5c72d,1,1,51.73,66.89,0.485437,0.073307,51.73,129.306012


In [643]:
# Validate the derived product-performance measures.
product_performance_measure_columns = ['product_item_contribution_within_category_percentage', 'product_gmv_contribution_within_category_percentage', 'product_average_item_price', 'product_freight_burden_percentage']

print("product_performance_measure_columns datatype:", "\n", product_performance[product_performance_measure_columns].dtypes)
print("\n")
print("product_performance_measure_columns missing value count:", "\n", product_performance[product_performance_measure_columns].isna().sum())
print("\n")
print("product_performance_measure_columns infinite value count:", "\n", (np.isinf(product_performance[product_performance_measure_columns])).sum())
print("\n")

product_performance_measure_columns datatype: 
 product_item_contribution_within_category_percentage    float64
product_gmv_contribution_within_category_percentage     float64
product_average_item_price                              float64
product_freight_burden_percentage                       float64
dtype: object


product_performance_measure_columns missing value count: 
 product_item_contribution_within_category_percentage    0
product_gmv_contribution_within_category_percentage     0
product_average_item_price                              0
product_freight_burden_percentage                       0
dtype: int64


product_performance_measure_columns infinite value count: 
 product_item_contribution_within_category_percentage    0
product_gmv_contribution_within_category_percentage     0
product_average_item_price                              0
product_freight_burden_percentage                       0
dtype: int64




In [644]:
print("product_item_contribution_within_category_percentage", "\n","Minimum value:", product_performance['product_item_contribution_within_category_percentage'].min(), "\n","Maximum value:",product_performance['product_item_contribution_within_category_percentage'].max() )
print("\n")
print("product_gmv_contribution_within_category_percentage", "\n","Minimum value:", product_performance['product_gmv_contribution_within_category_percentage'].min(), "\n","Maximum value:",product_performance['product_gmv_contribution_within_category_percentage'].max() )
print("\n")
print("product_average_item_price", "\n","Minimum value:", product_performance['product_average_item_price'].min(), "\n","Maximum value:",product_performance['product_average_item_price'].max() )
print("\n")
print("product_freight_burden_percentage", "\n","Minimum value:", product_performance['product_freight_burden_percentage'].min(), "\n","Maximum value:",product_performance['product_freight_burden_percentage'].max() )

product_item_contribution_within_category_percentage 
 Minimum value: 0.009129918743723181 
 Maximum value: 100.0


product_gmv_contribution_within_category_percentage 
 Minimum value: 0.00017840754270760306 
 Maximum value: 100.0


product_average_item_price 
 Minimum value: 0.85 
 Maximum value: 6735.0


product_freight_burden_percentage 
 Minimum value: 0.008849557522123894 
 Maximum value: 2304.3137254901962


In [645]:
# Inspect products whose freight value exceeds their merchandise value.
product_freight_burden_greater_than_100= product_performance[product_performance['product_freight_burden_percentage']>100].sort_values(by='product_freight_burden_percentage', ascending=False)
print("count of products with product_freight_burden_greater_than_100:", product_freight_burden_greater_than_100['product_id'].count())
product_freight_burden_greater_than_100[['product_category_name_english', 'product_id', 'product_item_count', 'product_gmv', 'product_freight_value', 'product_average_item_price', 'product_freight_burden_percentage']].head()


count of products with product_freight_burden_greater_than_100: 1174


,product_category_name_english,product_id,product_item_count,product_gmv,product_freight_value,product_average_item_price,product_freight_burden_percentage
8652,construction_tools_construction,8a3254bee785a526d548a81a9bc3c9be,3,2.55,58.76,0.85,2304.313725
15922,health_beauty,270516a3f41dc035aa87d220228f844c,10,12.00,78.90,1.20,657.500000
15591,health_beauty,05b515fdc76e888aada3c6d66c201dff,10,12.00,78.90,1.20,657.500000
19116,housewares,29781581fb82fe2389560a3a5331d0ee,3,9.18,54.69,3.06,595.751634
24491,signaling_and_security,bdbcda578f0215903f8bcd77ca591e0c,1,5.95,34.15,5.95,573.949580


**Product-performance table validation summary**

The product-performance table contains 32,216 distinct delivered products with no missing or duplicated product identifiers. Product item counts, GMV, and freight value reconcile fully with the delivered-item dataset, while product item and GMV contributions sum to approximately 100% within every category. All derived measures are complete and finite. A total of 1,174 products have freight value exceeding merchandise value, primarily among very low-priced products; these records are retained as commercially meaningful observations rather than treated as data errors.

### 5.5 Evaluate Product Performance and Identify Low-Performance Candidates

Products are evaluated within their assigned categories because sales volumes differ substantially across category sizes. Product-level item contribution and GMV contribution are used to identify comparatively weak products without allowing large categories to dominate the assessment. Products with unavailable category information are retained in the complete table but excluded from business rankings and low-performance classification.

In [646]:
# Rank available products by delivered item count within their categories.
available_product_mask = (product_performance['product_category_name_english'] != 'missing')

product_performance.loc[available_product_mask, 'product_item_count_rank_within_category'] = (product_performance.loc[available_product_mask].groupby('product_category_name_english')['product_item_count'].rank(method='dense', ascending=False))

In [647]:
# Validate product item-count rankings within available categories.
print("Missing rank count:", product_performance['product_item_count_rank_within_category'].isna().sum())
print("Available products with missing ranks:", len(product_performance[(product_performance['product_category_name_english']!='missing') & (product_performance['product_item_count_rank_within_category'].isna())]))
print("missing-category products with non-missing ranks:", len(product_performance[(product_performance['product_category_name_english']=='missing') & (product_performance['product_item_count_rank_within_category'].notna())]))
print("Minimum rank within every available category is 1: ", (np.isclose(product_performance.loc[available_product_mask].groupby('product_category_name_english')['product_item_count_rank_within_category'].min() ,1)).all())
print("product_item_count_rank_within_category datatype:", product_performance['product_item_count_rank_within_category'].dtype)

Missing rank count: 595
Available products with missing ranks: 0
missing-category products with non-missing ranks: 0
Minimum rank within every available category is 1:  True
product_item_count_rank_within_category datatype: float64


In [648]:
# Rank available products by GMV within their categories.
product_performance.loc[available_product_mask, 'product_gmv_rank_within_category'] = (product_performance.loc[available_product_mask].groupby('product_category_name_english')['product_gmv'].rank(method='dense', ascending=False))


In [649]:
# Validate product GMV rankings within available categories.
print("Missing product_gmv_rank_within_category count:", product_performance['product_gmv_rank_within_category'].isna().sum())
print("Available products with missing GMV ranks:", len(product_performance[(product_performance['product_category_name_english']!='missing') & (product_performance['product_gmv_rank_within_category'].isna())]))
print("missing-category products with non-missing GMV ranks:", len(product_performance[(product_performance['product_category_name_english']=='missing') & (product_performance['product_gmv_rank_within_category'].notna())]))
print("Minimum GMV rank within every available category is 1: ", (np.isclose(product_performance.loc[available_product_mask].groupby('product_category_name_english')['product_gmv_rank_within_category'].min() ,1)).all())
print("product_gmv_rank_within_category datatype:", product_performance['product_gmv_rank_within_category'].dtype)


Missing product_gmv_rank_within_category count: 595
Available products with missing GMV ranks: 0
missing-category products with non-missing GMV ranks: 0
Minimum GMV rank within every available category is 1:  True
product_gmv_rank_within_category datatype: float64


In [650]:
# Examine product counts by category before defining product-level performance thresholds.
category_performance_without_missing_category_name['category_product_count'].describe()

count      71.000000
mean      445.366197
std       728.263399
min         1.000000
25%        34.500000
50%        92.000000
75%       450.000000
max      2991.000000
Name: category_product_count, dtype: float64

In [651]:
#category name and product count for categories with fewer than four products.
category_performance_without_missing_category_name_fewer_than_4 = category_performance_without_missing_category_name[category_performance_without_missing_category_name['category_product_count']<4]
category_performance_without_missing_category_name_fewer_than_4[['product_category_name_english', 'category_product_count']]

,product_category_name_english,category_product_count
11,cds_dvds_musicals,1
62,security_and_services,2


In [652]:
# Define products eligible for within-category quartile screening.
product_performance['category_product_count'] = (product_performance.groupby('product_category_name_english')['product_id'].transform('count'))
product_performance['product_quartile_eligible'] = (product_performance['product_category_name_english']!='missing') & (product_performance['category_product_count']>=4)

In [653]:
# Validate product eligibility for within-category quartile screening.
eligible_category = product_performance[product_performance['product_quartile_eligible']==True]
print("Eligible categories — count distinct category names where product_quartile_eligible is True:", eligible_category['product_category_name_english'].nunique())
print("Eligible products — sum the Boolean column:", product_performance['product_quartile_eligible'].sum())
print("Available but ineligible products: category is not missing and product_quartile_eligible is False:",((product_performance['product_category_name_english']!='missing') & (product_performance['product_quartile_eligible']==False)).sum())
print("Eligible products in the missing category:", ((product_performance['product_category_name_english']=='missing') & (product_performance['product_quartile_eligible']==True)).sum())
print("product_quartile_eligible datatype:", product_performance['product_quartile_eligible'].dtype)

Eligible categories — count distinct category names where product_quartile_eligible is True: 69
Eligible products — sum the Boolean column: 31618
Available but ineligible products: category is not missing and product_quartile_eligible is False: 3
Eligible products in the missing category: 0
product_quartile_eligible datatype: bool


In [654]:
# Flag eligible products in the bottom item-count quartile within their categories.
product_performance.loc[product_performance['product_quartile_eligible'], 'product_item_count_q1_within_category'] = product_performance.loc[product_performance['product_quartile_eligible']].groupby('product_category_name_english')['product_item_count'].transform('quantile', q=0.25)


In [655]:
# Identify eligible products at or below their category’s item-count 25th percentile.
product_performance['low_product_volume_candidate'] = (product_performance['product_quartile_eligible']==True) & (product_performance['product_item_count']<=product_performance['product_item_count_q1_within_category'])

In [656]:
# Validate the within-category low-volume product classification.
print("Missing threshold values:", product_performance['product_item_count_q1_within_category'].isna().sum())
print("Eligible products with missing thresholds:", ((product_performance['product_quartile_eligible']==True) & (product_performance['product_item_count_q1_within_category'].isna())).sum())
print("Ineligible products marked as low-volume candidates:", ((product_performance['product_quartile_eligible']==False) & (product_performance['low_product_volume_candidate']==True)).sum())

low_product_volume_candidates=product_performance[(product_performance['product_quartile_eligible']==True) & (product_performance['product_item_count']<=product_performance['product_item_count_q1_within_category'])]
print("Every low-volume candidate has product_item_count <= product_item_count_q1_within_category: ", ((low_product_volume_candidates['low_product_volume_candidate']==True) & (low_product_volume_candidates['product_item_count'] <= low_product_volume_candidates['product_item_count_q1_within_category'])).all())
print("Missing values in low_product_volume_candidate:", product_performance['low_product_volume_candidate'].isna().sum())
print("low_product_volume_candidate data type:", product_performance['low_product_volume_candidate'].dtype)
print("Number of low-volume candidates:",len(low_product_volume_candidates))

Missing threshold values: 598
Eligible products with missing thresholds: 0
Ineligible products marked as low-volume candidates: 0
Every low-volume candidate has product_item_count <= product_item_count_q1_within_category:  True
Missing values in low_product_volume_candidate: 0
low_product_volume_candidate data type: bool
Number of low-volume candidates: 17345


In [657]:
# Calculate the category-specific GMV 25th percentile for eligible products.
product_performance.loc[product_performance['product_quartile_eligible'], 'product_gmv_q1_within_category'] = product_performance.loc[product_performance['product_quartile_eligible']].groupby('product_category_name_english')['product_gmv'].transform('quantile', q=0.25)

In [658]:
# Identify eligible products at or below their category’s GMV 25th percentile.
product_performance['low_product_gmv_candidate'] = (product_performance['product_quartile_eligible']==True) & (product_performance['product_gmv']<=product_performance['product_gmv_q1_within_category'])


In [659]:
# Validate the within-category low-GMV product classification.
print("Missing product_gmv_q1_within_category values:", product_performance['product_gmv_q1_within_category'].isna().sum())
print("Eligible products with missing GMV thresholds:", ((product_performance['product_quartile_eligible']==True) & (product_performance['product_gmv_q1_within_category'].isna())).sum())
print("Ineligible products marked as low-GMV candidates:", ((product_performance['product_quartile_eligible']==False) & (product_performance['low_product_gmv_candidate']==True)).sum())

low_product_gmv_candidates = product_performance[product_performance['low_product_gmv_candidate'] == True]
print("Every low-GMV candidate has GMV at or below its category threshold:",(low_product_gmv_candidates['product_gmv']<= low_product_gmv_candidates['product_gmv_q1_within_category']).all())
print("Missing values in low_product_gmv_candidate:", product_performance['low_product_gmv_candidate'].isna().sum())
print("low_product_gmv_candidate data type:", product_performance['low_product_gmv_candidate'].dtype)
print("Number of low-GMV candidates:", len(low_product_gmv_candidates))

Missing product_gmv_q1_within_category values: 598
Eligible products with missing GMV thresholds: 0
Ineligible products marked as low-GMV candidates: 0
Every low-GMV candidate has GMV at or below its category threshold: True
Missing values in low_product_gmv_candidate: 0
low_product_gmv_candidate data type: bool
Number of low-GMV candidates: 7995


In [660]:
# Flag products that fall in the bottom within-category quartile for both item volume and GMV.
product_performance['low_product_performance_candidate'] = (product_performance['low_product_volume_candidate']==True) & (product_performance['low_product_gmv_candidate']==True)

In [661]:
# Validate the final low-performance product classification.
print("Number of final candidates:", product_performance['low_product_performance_candidate'].sum())
print("Ineligible products marked as final candidates:", ((product_performance['product_quartile_eligible'] == False) & (product_performance['low_product_performance_candidate'] == True)).sum())
print("Every final candidate has both component flags equal to True:", (product_performance['low_product_performance_candidate']==((product_performance['low_product_volume_candidate']==True) & (product_performance['low_product_gmv_candidate']==True))).all())

final_low_product_candidates = product_performance[product_performance['low_product_performance_candidate'] == True]
print("Every final candidate is at or below both its item-count and GMV thresholds:", ((final_low_product_candidates['product_item_count'] <= final_low_product_candidates['product_item_count_q1_within_category']) & (final_low_product_candidates['product_gmv']<= final_low_product_candidates['product_gmv_q1_within_category'])).all())

print("low_product_performance_candidate missing values:", product_performance['low_product_performance_candidate'].isna().sum())
print("low_product_performance_candidate data type of :", product_performance['low_product_performance_candidate'].dtype)


Number of final candidates: 6779
Ineligible products marked as final candidates: 0
Every final candidate has both component flags equal to True: True
Every final candidate is at or below both its item-count and GMV thresholds: True
low_product_performance_candidate missing values: 0
low_product_performance_candidate data type of : bool


In [664]:
# Create the final product-level low-performance candidate table.
low_performance_product_candidates = product_performance[product_performance['low_product_performance_candidate']==True].copy()
low_performance_product_candidates = low_performance_product_candidates.sort_values(by=['product_category_name_english','product_item_count', 'product_gmv'], ascending=True).reset_index(drop=True)
low_performance_product_candidates.head()

,product_category_name_english,product_id,product_item_count,product_order_count,product_gmv,product_freight_value,product_item_contribution_within_category_percentage,product_gmv_contribution_within_category_percentage,product_average_item_price,product_freight_burden_percentage,product_item_count_rank_within_category,product_gmv_rank_within_category,category_product_count,product_quartile_eligible,product_item_count_q1_within_category,low_product_volume_candidate,product_gmv_q1_within_category,low_product_gmv_candidate,low_product_performance_candidate
0,agro_industry_and_commerce,8a67e4e6cc6f2abf65fe2164ea51b219,1,1,12.99,15.23,0.485437,0.018408,12.99,117.244034,11.0,69.0,71,True,1.0,True,116.575,True,True
1,agro_industry_and_commerce,ac4bd2aec3f990363aa853e4f17709d3,1,1,13.00,7.39,0.485437,0.018422,13.00,56.846154,11.0,68.0,71,True,1.0,True,116.575,True,True
2,agro_industry_and_commerce,a6bfd5412c292e6ab465ba7d7f794727,1,1,22.00,14.52,0.485437,0.031176,22.00,66.000000,11.0,67.0,71,True,1.0,True,116.575,True,True
3,agro_industry_and_commerce,137ace556a03792cdc43f91ec621426d,1,1,23.90,18.27,0.485437,0.033869,23.90,76.443515,11.0,66.0,71,True,1.0,True,116.575,True,True
4,agro_industry_and_commerce,613d093272cb8f74f25a01e430155a6a,1,1,29.50,9.94,0.485437,0.041805,29.50,33.694915,11.0,65.0,71,True,1.0,True,116.575,True,True


In [671]:
# Validate the final product-level low-performance candidate table.
print("low_performance_product_candidates shape:", low_performance_product_candidates.shape)
print("Duplicate product_id values:", low_performance_product_candidates['product_id'].duplicated().sum())
print("Missing values across the table:", low_performance_product_candidates.isna().sum().sum())
print("All product_quartile_eligible values:", low_performance_product_candidates['product_quartile_eligible'].all())
print("All low_product_performance_candidate values:", low_performance_product_candidates['low_product_performance_candidate'].all())
print("Every row satisfies both item-count and GMV thresholds:", ((low_performance_product_candidates['product_item_count'] <= low_performance_product_candidates['product_item_count_q1_within_category']) & (low_performance_product_candidates['product_gmv']<= low_performance_product_candidates['product_gmv_q1_within_category'])).all())
print("Number of represented categories:", low_performance_product_candidates['product_category_name_english'].nunique())

low_performance_product_candidates shape: (6779, 19)
Duplicate product_id values: 0
Missing values across the table: 0
All product_quartile_eligible values: True
All low_product_performance_candidate values: True
Every row satisfies both item-count and GMV thresholds: True
Number of represented categories: 69


**Low-performance product screening summary**

Of the 31,618 products eligible for within-category comparison, 6,779 products—approximately 21.4%—fall at or below their category-specific 25th percentile for both delivered-item volume and GMV. These candidates span all 69 eligible categories. Products with unavailable category information and products from categories containing fewer than four products were excluded from quartile screening but retained in the complete product-performance table. The resulting classification is a screening indicator rather than definitive evidence of poor performance because product availability periods, inventory levels, promotional exposure, costs, and profitability are unavailable.

## 6. Validate and Save Day 5 KPI Tables

The completed overall, monthly, category, and product KPI tables are reconciled against the analysis-ready delivered-order-item dataset before export. This final validation confirms that additive measures remain consistent across analytical grains and that no duplication or omission was introduced during aggregation.

In [ ]:
# Reconcile delivered-order totals between the master and monthly tables.
master_delivered_orders = category_analysis_items['order_id'].nunique()
monthly_orders_total = monthly_kpi_summary['monthly_order_count'].sum()

print("Master and monthly delivered-order totals reconcile:",master_delivered_orders == monthly_orders_total)

print("Master delivered orders:", master_delivered_orders)
print("Monthly delivered orders:", monthly_orders_total)



Master and monthly delivered-order totals reconcile: True
Master delivered orders: 96478
Monthly delivered orders: 96478


In [719]:
# Reconcile delivered-items totals across the master, monthly, category, and product tables.
master_delivered_items = category_analysis_items['order_item_id'].count()
monthly_items_total = monthly_kpi_summary['monthly_item_count'].sum()
category_items_total = category_performance['category_item_count'].sum()
product_items_total = product_performance['product_item_count'].sum()

print("Master and monthly delivered-item totals reconcile:", master_delivered_items == monthly_items_total)
print("Master delivered item:", master_delivered_items)
print("Monthly delivered items:", monthly_items_total)
print("\n")

print("Master and category delivered-item totals reconcile:", master_delivered_items == category_items_total)
print("Master delivered item:", master_delivered_items)
print("Category delivered items:", category_items_total)
print("\n")

print("Master and product delivered-item totals reconcile:", master_delivered_items == product_items_total)
print("Master delivered item:", master_delivered_items)
print("Product delivered items:", product_items_total)


Master and monthly delivered-item totals reconcile: True
Master delivered item: 110197
Monthly delivered items: 110197


Master and category delivered-item totals reconcile: True
Master delivered item: 110197
Category delivered items: 110197


Master and product delivered-item totals reconcile: True
Master delivered item: 110197
Product delivered items: 110197


In [ ]:
# Reconcile GMV totals across the master, monthly, category, and product tables.
master_gmv = category_analysis_items['price'].sum()
monthly_gmv_total = monthly_kpi_summary['monthly_gmv'].sum()
category_gmv_total = category_performance['category_gmv'].sum()
product_gmv_total = product_performance['product_gmv'].sum()

print("Master and Monthly gmv totals reconcile:", np.isclose(master_gmv,monthly_gmv_total))
print("Master gmv:", master_gmv)
print("Monthly gmv:", monthly_gmv_total)
print("\n")

print("Master and Category gmv totals reconcile:", np.isclose(master_gmv,category_gmv_total))
print("Master gmv:", master_gmv)
print("Category gmv:", category_gmv_total)
print("\n")

print("Master and Product gmv totals reconcile:", np.isclose(master_gmv,product_gmv_total))
print("Master gmv:", master_gmv)
print("Product gmv:", product_gmv_total)

Master and Monthly gmv totals reconcile: True
Master gmv: 13221498.11
Monthly gmv: 13221498.11


Master and Category gmv totals reconcile: True
Master gmv: 13221498.11
Category gmv: 13221498.11


Master and Product gmv totals reconcile: True
Master gmv: 13221498.11
Product gmv: 13221498.11


In [722]:
# Reconcile freight totals across the master, monthly, category, and product tables.
master_freight_total = category_analysis_items['freight_value'].sum()
monthly_freight_total = monthly_kpi_summary['monthly_freight_value'].sum()
category_freight_total = category_performance['category_freight_value'].sum()
product_freight_total = product_performance['product_freight_value'].sum()

print("Master and Monthly freight-value totals reconcile:", np.isclose(master_freight_total,monthly_freight_total))
print("Master freight-value:", master_freight_total)
print("Monthly freight-value:", monthly_freight_total)
print("\n")

print("Master and Category freight-value totals reconcile:", np.isclose(master_freight_total,category_freight_total))
print("Master freight-value:", master_freight_total)
print("Category freight-value:", category_freight_total)
print("\n")

print("Master and Product freight-value totals reconcile:", np.isclose(master_freight_total,product_freight_total))
print("Master freight-value:", master_freight_total)
print("Product freight-value:", product_freight_total)

Master and Monthly freight-value totals reconcile: True
Master freight-value: 2198275.64
Monthly freight-value: 2198275.6399999997


Master and Category freight-value totals reconcile: True
Master freight-value: 2198275.64
Category freight-value: 2198275.64


Master and Product freight-value totals reconcile: True
Master freight-value: 2198275.64
Product freight-value: 2198275.6399999997


**Cross-table reconciliation summary**

Delivered-order, delivered-item, GMV, and freight totals reconcile successfully between the analysis-ready master dataset and the monthly, category, and product KPI tables. Minor additional decimal places in some freight totals result from floating-point representation and do not indicate a discrepancy. These checks confirm that the aggregations preserved the complete delivered-order scope without duplicating or omitting additive measures.

In [723]:
# Save the validated Day 5 KPI and performance tables.

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

output_directory = project_root / "reports" / "tables"
output_directory.mkdir(parents=True, exist_ok=True)

# Prepare the overall KPI summary for CSV export.
if isinstance(overall_kpi_summary, pd.Series):
    overall_kpi_export = overall_kpi_summary.rename("value").reset_index()
    overall_kpi_export.columns = ["metric", "value"]
elif isinstance(overall_kpi_summary, dict):
    overall_kpi_export = pd.DataFrame(overall_kpi_summary.items(), columns=["metric", "value"])
else:
    overall_kpi_export = overall_kpi_summary.copy()

overall_kpi_export.to_csv(output_directory/"overall_kpi_summary.csv",index=False)

monthly_kpi_summary.to_csv(output_directory/"monthly_kpi_summary.csv",index=False)

category_performance.to_csv(output_directory/"category_performance.csv",index=False)

low_performance_category_candidates.to_csv(output_directory/"low_performance_category_candidates.csv",index=False)

product_performance.to_csv(output_directory/"product_performance.csv",index=False)

low_performance_product_candidates.to_csv(output_directory/"low_performance_product_candidates.csv",index=False)

print("Validated Day 5 tables saved to:", output_directory.resolve())

Validated Day 5 tables saved to: C:\Users\Hp\DataAnalyst\reports\tables


In [726]:
# Read back one exported Day 5 table for validation.
overall_kpi_export_check = pd.read_csv(output_directory/"overall_kpi_summary.csv")

monthly_kpi_summary_export_check = pd.read_csv(output_directory/"monthly_kpi_summary.csv")

category_performance_export_check = pd.read_csv(output_directory/"category_performance.csv")

product_performance_export_check = pd.read_csv(output_directory/"product_performance.csv")

low_performance_category_candidates_export_check = pd.read_csv(output_directory/"low_performance_category_candidates.csv")

low_performance_product_candidates_export_check = pd.read_csv(output_directory/"low_performance_product_candidates.csv")

In [728]:
print("overall_kpi_export_check shape:", overall_kpi_export_check.shape)
print("\n")
print("monthly_kpi_summary_export_check shape:", monthly_kpi_summary_export_check.shape)
print("\n")
print("category_performance_export_check shape:", category_performance_export_check.shape)
print("\n")
print("product_performance_export_check shape:", product_performance_export_check.shape)
print("\n")
print("low_performance_category_candidates_export_check shape:", low_performance_category_candidates_export_check.shape)
print("\n")
print("low_performance_product_candidates_export_check shape:", low_performance_product_candidates_export_check.shape)


overall_kpi_export_check shape: (25, 2)


monthly_kpi_summary_export_check shape: (24, 24)


category_performance_export_check shape: (72, 14)


product_performance_export_check shape: (32216, 19)


low_performance_category_candidates_export_check shape: (17, 14)


low_performance_product_candidates_export_check shape: (6779, 19)


In [729]:
# Confirm that exported tables preserve the original column structures.
print("Overall KPI columns preserved:", overall_kpi_export_check.columns.equals(overall_kpi_export.columns))

print("Monthly KPI columns preserved:", monthly_kpi_summary_export_check.columns.equals(monthly_kpi_summary.columns))

print("Category-performance columns preserved:", category_performance_export_check.columns.equals(category_performance.columns))

print("Product-performance columns preserved:", product_performance_export_check.columns.equals(product_performance.columns))

print("Low-performance category columns preserved:", low_performance_category_candidates_export_check.columns.equals(low_performance_category_candidates.columns))

print("Low-performance product columns preserved:", low_performance_product_candidates_export_check.columns.equals(low_performance_product_candidates.columns))

Overall KPI columns preserved: True
Monthly KPI columns preserved: True
Category-performance columns preserved: True
Product-performance columns preserved: True
Low-performance category columns preserved: True
Low-performance product columns preserved: True


### Export Validation Summary

All six Day 5 output tables were successfully exported to the `reports/tables` directory and read back without errors. Their exported row and column counts match the original tables, and all column structures were preserved. Together with the completed cross-table reconciliation, these checks confirm that the KPI and performance outputs are complete and ready for subsequent analysis, visualization, and reporting.